# p = 1000: post-processing, tables and figures

Run after the five training notebooks and the Gibbs notebook. Evaluates every configuration, selects one fit per method and dataset by the evaluated ELBO rule, applies Ordinary PSIS, IWMM and Iterative Damped PSIS with $K=5000$, and scores the four stages against the Gibbs reference (`output_gibbs_tau2_1`) and the generating truth. Results (including ESS) go to `p1000_POSTPROCESSING_tau2_1_final_S1_V20/<run id>`.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Mapping, Optional, Tuple
import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
import json
import math
import zlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.special import betaln
try:
    import arviz as az
except ImportError:
    az = None
try:
    from arviz_stats.base.array import array_stats as _arviz_array_stats
except ImportError:
    _arviz_array_stats = None
K_FIXED = 5000
PARETO_K_STOP = 0.7
IWMM_MAX_ITERS = 20
DAMPED_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
DAMPED_MIN_K_IMPROVEMENT = 1e-12
PSIS_BASE_SEED = 1217
DTYPE = torch.float64
DEVICE = torch.device('cpu')
METHOD_CODE = {'ELBO': 11, 'VIMCO*': 23, 'VIMCO* annealed': 37, 'VIMCO-GM': 41, 'VIMCO-GM annealed': 53}
PROBABILITY_METRIC_TOL = 1e-10
import hashlib
from scipy.special import expit
from scipy.stats import t as student_t
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
ROOT = Path.cwd().resolve()
DATA_DIR_CANDIDATES = (ROOT / 'data',)
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.is_dir()), DATA_DIR_CANDIDATES[0])
OUTPUT_ROOT = ROOT / 'p1000_POSTPROCESSING_tau2_1_final_S1_V20'

def safe_method_name(method):
    return method.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()

TRAINING_DIRS = {name: ROOT / folder for name,folder in [('ELBO','output_elbo_tau2_1'),('VIMCO*','output_star_tau2_1'),('VIMCO* annealed','output_star_annealed_tau2_1'),('VIMCO-GM','output_GM_tau2_1'),('VIMCO-GM annealed','output_GM_annealed_tau2_1')]}
DATA_SEEDS = range(50)
EXPECTED_SEEDS = tuple(DATA_SEEDS)
TAU2 = 1.0
P = 1000
A_PI, B_PI = (0.02, 1.0)
METHODS = list(TRAINING_DIRS)
STAGES = ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']
METRICS = ['PIP RMSE vs Gibbs', 'Mean excess marginal FKL vs Gibbs', 'PIP RMSE vs truth', 'F1 vs truth', 'Mean marginal FKL vs truth', 'Pareto k-hat', '95% latent coverage', '95% interval score']
EVAL_ELBO_SAMPLES = 2000
EVAL_ELBO_SEED = 2004
EVAL_TARGET_VERSION = 'p1000_tau2_1_v20_clean'
N_XSTAR = 100
N_MIXTURE_DRAWS = 20000
AR1_RHO = 0.8
BATCH_SIZE = 250
ADAPTED_PROB_EPS = 1e-12
TORCH_THREADS = 2
ANALYSIS_VERSION = 'p1000_V20_truth_selected250_cutoff80_batch250_v1'

In [ ]:
def stable_method_code(method: str) -> int:
    return METHOD_CODE.get(str(method), 97)

def stable_seed(seed: int, method: str, setting_label: str, tag: str='') -> int:
    base = PSIS_BASE_SEED + int(seed)
    if tag in ('', '|ORDINARY_SHARED'):
        return base
    if tag.startswith('|DAMPED_PILOT'):
        stage = 1
    elif tag.startswith('|DAMPED_VALIDATE'):
        stage = 2
    elif tag.startswith('|IWMM_CANDIDATE'):
        stage = 3
    elif tag.startswith('|DAMPED_FINAL') or tag == '|IWMM_FINAL_EVAL':
        stage = 4
    else:
        raise ValueError(f'Unknown PSIS stage tag: {tag}')
    iteration = int(tag.split('|iter=')[-1].split('|')[0]) if '|iter=' in tag and stage <= 3 else 1
    if not 1 <= iteration <= IWMM_MAX_ITERS:
        raise ValueError(f'Invalid PSIS iteration: {iteration}')
    return base + 100000 * stage + 1000 * (iteration - 1)

def _normalise_logweights(logw: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(logw, dtype=np.float64).reshape(-1)
    if x.size == 0 or not np.all(np.isfinite(x)):
        raise FloatingPointError('Importance ratios must be finite and non-empty.')
    m = float(np.max(x))
    log_norm = m + math.log(float(np.sum(np.exp(x - m))))
    lw = x - log_norm
    w = np.exp(lw)
    w /= w.sum()
    return (lw, w)

def psis_smooth_logweights(raw_log_p_over_q: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    global _PSIS_LAST_STATUS
    _PSIS_LAST_STATUS = 'ok'
    x = np.asarray(raw_log_p_over_q, dtype=np.float64).reshape(-1)
    if x.size < 10 or not np.all(np.isfinite(x)):
        raise ValueError('PSIS requires at least 10 finite raw log importance ratios.')
    try:
        if az is not None and hasattr(az, 'psislw'):
            lw, khat = az.psislw(x, reff=1.0)
        elif _arviz_array_stats is not None:
            lw, khat = _arviz_array_stats.psislw(-x, r_eff=1.0, axis=-1)
        else:
            raise ImportError('A maintained ArviZ PSIS implementation is required. Install/update arviz.')
    except ValueError as exc:
        if 'tail values are the same' not in str(exc).lower():
            raise
        lw, w = _normalise_logweights(x)
        _PSIS_LAST_STATUS = 'fit_unavailable_equal_tail_raw_weights'
        return (lw, w, float('nan'))
    lw = np.asarray(lw, dtype=np.float64).reshape(-1)
    khat_arr = np.asarray(khat, dtype=np.float64).reshape(-1)
    if lw.shape != x.shape or khat_arr.size != 1:
        raise RuntimeError('Unexpected PSIS output shape.')
    _, w = _normalise_logweights(lw)
    if not np.isfinite(khat_arr[0]):
        _PSIS_LAST_STATUS = 'fit_unavailable_nonfinite_k'
    return (lw, w, float(khat_arr[0]))

def validate_psis_backend_sign() -> None:
    x = np.linspace(-4.0, 4.0, 401, dtype=np.float64)
    _, w, _ = psis_smooth_logweights(x)
    if not np.isclose(w.sum(), 1.0, atol=1e-12):
        raise RuntimeError('PSIS backend sign self-test failed: weights do not sum to one.')
    if not float(w[-40:].sum()) > float(w[:40].sum()):
        raise RuntimeError('PSIS backend sign self-test failed: ratio ordering reversed.')

def sigmoid_eta_exact(eta: np.ndarray, label: str='eta') -> np.ndarray:
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    if eta.size == 0 or not np.all(np.isfinite(eta)):
        raise ValueError(f'{label}: logits must be finite and non-empty.')
    rho = torch.sigmoid(torch.from_numpy(eta)).numpy()
    if np.any(rho <= 0.0) or np.any(rho >= 1.0):
        raise FloatingPointError(f'{label}: saved logits imply numerically degenerate probabilities. The original proposal is not clipped in the audited comparison.')
    return rho.astype(np.float64, copy=False)

def adapted_prob_to_eta(rho: np.ndarray, eps: float, label: str='adapted proposal') -> Tuple[np.ndarray, np.ndarray]:
    rho = np.asarray(rho, dtype=np.float64).reshape(-1)
    if rho.size == 0 or not np.all(np.isfinite(rho)):
        raise ValueError(f'{label}: probabilities must be finite and non-empty.')
    if np.any(rho < -1e-12) or np.any(rho > 1.0 + 1e-12):
        raise ValueError(f'{label}: probabilities outside [0,1].')
    eps = float(eps)
    if not 0.0 < eps < 0.5:
        raise ValueError('Adaptation clipping epsilon must lie in (0,0.5).')
    clipped = np.clip(rho, eps, 1.0 - eps)
    eta = np.log(clipped) - np.log1p(-clipped)
    if not np.all(np.isfinite(eta)):
        raise FloatingPointError(f'{label}: non-finite adapted logits.')
    return (clipped, eta)

def _sanitize_probability_vector(x: np.ndarray, label: str='probability vector', tol: float=PROBABILITY_METRIC_TOL) -> np.ndarray:
    a = np.asarray(x, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(a)):
        raise ValueError(f'{label} must be finite.')
    lo = float(np.min(a)) if a.size else 0.0
    hi = float(np.max(a)) if a.size else 1.0
    if lo < -tol or hi > 1.0 + tol:
        raise ValueError(f'{label} lies outside [0,1] beyond numerical tolerance {tol:g}: min={lo:.17g}, max={hi:.17g}.')
    return np.clip(a, 0.0, 1.0)

def forward_kl_product_bernoulli(reference_pip: np.ndarray, approx_pip: np.ndarray) -> float:
    pi_raw = np.asarray(reference_pip, dtype=np.float64).reshape(-1)
    rho_raw = np.asarray(approx_pip, dtype=np.float64).reshape(-1)
    if pi_raw.shape != rho_raw.shape:
        raise ValueError('FKL inputs must have the same shape.')
    pi = _sanitize_probability_vector(pi_raw, 'FKL reference PIPs')
    rho = _sanitize_probability_vector(rho_raw, 'FKL approximate PIPs')
    eps = 1e-12
    r = np.clip(rho, eps, 1.0 - eps)
    term1 = np.where(pi > 0.0, pi * (np.log(np.clip(pi, eps, 1.0)) - np.log(r)), 0.0)
    one_pi = 1.0 - pi
    term0 = np.where(one_pi > 0.0, one_pi * (np.log(np.clip(one_pi, eps, 1.0)) - np.log1p(-r)), 0.0)
    out = float(np.sum(term1 + term0))
    if not np.isfinite(out) or out < -1e-10:
        raise RuntimeError(f'Computed FKL is invalid: {out}.')
    return max(out, 0.0)

@dataclass(frozen=True)
class TargetSpec:
    tau2: float
    a_pi: float
    b_pi: float = 1.0
    jitter: float = 1e-08
    primal_active_max: int = 80
    primal_batch_size: int = 64
    dual_batch_size: int = 8

class CollapsedLinearTarget:

    def __init__(self, X: np.ndarray, y: np.ndarray, spec: TargetSpec):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        if X.ndim != 2 or y.ndim != 1 or X.shape[0] != len(y):
            raise ValueError('X must be n x p and y must be length n.')
        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            raise ValueError('X and y must be finite.')
        if X.shape[0] < 3 or X.shape[1] < 1:
            raise ValueError('Need n>=3 and p>=1.')
        if not (spec.tau2 > 0 and spec.a_pi > 0 and (spec.b_pi > 0) and (spec.jitter > 0)):
            raise ValueError('Target hyperparameters must be positive.')
        self.X = np.ascontiguousarray(X)
        self.y = np.ascontiguousarray(y)
        self.spec = spec
        self.n, self.p = X.shape
        self.nu = float(self.n - 1)
        self.prior_prec = 1.0 / float(spec.tau2)
        self.log_prior_prec = math.log(self.prior_prec)
        self.log_n = math.log(float(self.n))
        self.marginal_constant = float(math.lgamma(0.5 * self.nu) - 0.5 * self.nu * math.log(2.0 * math.pi))
        X_full = np.column_stack([np.ones(self.n, dtype=np.float64), self.X])
        self.XtX = torch.from_numpy(X_full.T @ X_full).to(dtype=DTYPE, device=DEVICE)
        self.Xty = torch.from_numpy(X_full.T @ self.y).to(dtype=DTYPE, device=DEVICE)
        self.yty = torch.tensor(float(self.y @ self.y), dtype=DTYPE, device=DEVICE)
        Xc = self.X - self.X.mean(axis=0, keepdims=True)
        yc = self.y - self.y.mean()
        self.Xc_T = torch.from_numpy(np.ascontiguousarray(Xc)).to(dtype=DTYPE, device=DEVICE).t().contiguous()
        self.yc = torch.from_numpy(np.ascontiguousarray(yc)).to(dtype=DTYPE, device=DEVICE)
        self.ycyc = torch.tensor(float(yc @ yc), dtype=DTYPE, device=DEVICE)
        self.I_n = torch.eye(self.n, dtype=DTYPE, device=DEVICE)
        k = np.arange(self.p + 1, dtype=np.float64)
        log_prior = betaln(k + spec.a_pi, self.p - k + spec.b_pi) - betaln(spec.a_pi, spec.b_pi)
        self.log_prior_by_size = torch.from_numpy(log_prior).to(dtype=DTYPE, device=DEVICE)
        self._regularisation: Dict[int, torch.Tensor] = {}

    def _reg_matrix(self, q: int) -> torch.Tensor:
        q = int(q)
        mat = self._regularisation.get(q)
        if mat is None:
            diag = torch.full((q,), self.prior_prec, dtype=DTYPE, device=DEVICE)
            diag[0] = 0.0
            mat = torch.diag(diag) + self.spec.jitter * torch.eye(q, dtype=DTYPE, device=DEVICE)
            self._regularisation[q] = mat
        return mat

    def _log_prior(self, G: torch.Tensor) -> torch.Tensor:
        k = G.sum(dim=1).to(torch.long)
        return self.log_prior_by_size[k]

    def _log_marginal_primal(self, G: torch.Tensor) -> torch.Tensor:
        B = int(G.shape[0])
        out = torch.empty(B, dtype=DTYPE, device=DEVICE)
        active_counts = G.sum(dim=1).to(torch.long)
        for count_t in torch.unique(active_counts):
            k = int(count_t.item())
            rows_all = torch.nonzero(active_counts == k, as_tuple=False).flatten()
            q = k + 1
            for start in range(0, int(rows_all.numel()), int(self.spec.primal_batch_size)):
                rows = rows_all[start:start + int(self.spec.primal_batch_size)]
                bsz = int(rows.numel())
                if k == 0:
                    idx = torch.zeros((bsz, 1), dtype=torch.long, device=DEVICE)
                else:
                    group = G.index_select(0, rows)
                    nz = torch.nonzero(group > 0.5, as_tuple=False)
                    slope_idx = nz[:, 1].reshape(bsz, k).to(torch.long) + 1
                    intercept = torch.zeros((bsz, 1), dtype=torch.long, device=DEVICE)
                    idx = torch.cat([intercept, slope_idx], dim=1)
                XtX_b = self.XtX[idx[:, :, None], idx[:, None, :]]
                Xty_b = self.Xty[idx]
                M = XtX_b + self._reg_matrix(q).unsqueeze(0)
                L = torch.linalg.cholesky(M)
                solved = torch.cholesky_solve(Xty_b[..., None], L).squeeze(-1)
                residual = self.yty - torch.sum(Xty_b * solved, dim=1)
                residual = torch.clamp(residual, min=self.spec.jitter)
                logdet_M = 2.0 * torch.sum(torch.log(torch.diagonal(L, dim1=-2, dim2=-1)), dim=1)
                out[rows] = self.marginal_constant + 0.5 * k * self.log_prior_prec - 0.5 * logdet_M - 0.5 * self.nu * torch.log(residual / 2.0)
        return out

    def _log_marginal_dual(self, G: torch.Tensor) -> torch.Tensor:
        B = int(G.shape[0])
        out = torch.empty(B, dtype=DTYPE, device=DEVICE)
        active_counts = G.sum(dim=1).to(torch.long)
        for count_t in torch.unique(active_counts):
            k = int(count_t.item())
            rows = torch.nonzero(active_counts == k, as_tuple=False).flatten()
            gsz = int(rows.numel())
            if k == 0:
                residual = torch.clamp(self.ycyc, min=self.spec.jitter)
                out[rows] = self.marginal_constant - 0.5 * self.log_n - 0.5 * self.nu * torch.log(residual / 2.0)
                continue
            group = G.index_select(0, rows)
            nz = torch.nonzero(group > 0.5, as_tuple=False)
            idx = nz[:, 1].reshape(gsz, k).to(torch.long)
            for start in range(0, gsz, int(self.spec.dual_batch_size)):
                current_rows = rows[start:start + int(self.spec.dual_batch_size)]
                current_idx = idx[start:start + int(self.spec.dual_batch_size)]
                bsz = int(current_idx.shape[0])
                XA = self.Xc_T[current_idx].transpose(1, 2).contiguous()
                Kmat = self.I_n.unsqueeze(0) + float(self.spec.tau2) * torch.bmm(XA, XA.transpose(1, 2))
                Kmat = Kmat + self.spec.jitter * self.I_n.unsqueeze(0)
                L = torch.linalg.cholesky(Kmat)
                yb = self.yc.expand(bsz, -1).unsqueeze(2)
                solved = torch.cholesky_solve(yb, L).squeeze(2)
                residual = torch.sum(self.yc.unsqueeze(0) * solved, dim=1)
                residual = torch.clamp(residual, min=self.spec.jitter)
                logdet_K = 2.0 * torch.sum(torch.log(torch.diagonal(L, dim1=-2, dim2=-1)), dim=1)
                out[current_rows] = self.marginal_constant - 0.5 * self.log_n - 0.5 * logdet_K - 0.5 * self.nu * torch.log(residual / 2.0)
        return out

    def log_joint(self, G: torch.Tensor) -> torch.Tensor:
        if G.ndim != 2 or int(G.shape[1]) != self.p:
            raise ValueError(f'G must have shape (B,{self.p}).')
        G = G.to(dtype=DTYPE, device=DEVICE)
        active = G.sum(dim=1)
        primal_mask = active <= int(self.spec.primal_active_max)
        out = torch.empty(int(G.shape[0]), dtype=DTYPE, device=DEVICE)
        if bool(primal_mask.any().item()):
            idx = torch.nonzero(primal_mask, as_tuple=False).flatten()
            out[idx] = self._log_marginal_primal(G.index_select(0, idx))
        if bool((~primal_mask).any().item()):
            idx = torch.nonzero(~primal_mask, as_tuple=False).flatten()
            out[idx] = self._log_marginal_dual(G.index_select(0, idx))
        return out + self._log_prior(G)

    def audit_primal_dual(self, G: np.ndarray, atol: float=1e-05) -> float:
        Gt = torch.as_tensor(np.asarray(G, dtype=np.float64), dtype=DTYPE, device=DEVICE)
        with torch.no_grad():
            a = self._log_marginal_primal(Gt)
            b = self._log_marginal_dual(Gt)
        d = (a - b).cpu().numpy()
        gap = float(np.max(np.abs(d))) if len(d) else 0.0
        if gap > float(atol):
            raise RuntimeError(f'Primal/dual collapsed-target audit failed: max absolute gap={gap:.3e}.')
        return gap

@dataclass
class DrawResult:
    eta: np.ndarray
    rho: np.ndarray
    G: np.ndarray
    raw_logw: np.ndarray
    raw_weights: np.ndarray
    psis_weights: np.ndarray
    pip: np.ndarray
    khat: float
    raw_ess: float
    psis_ess: float
    seed: int
    K: int

    psis_status: str = 'ok'

    def diagnostics(self) -> Dict[str, Any]:
        return {'psis_status': self.psis_status, 'n_unique_logweights': int(np.unique(self.raw_logw).size), 'K': int(self.K), 'seed': int(self.seed), 'Pareto_k': float(self.khat), 'raw_ESS': float(self.raw_ess), 'PSIS_ESS': float(self.psis_ess), 'PSIS_ESS_fraction': float(self.psis_ess / self.K), 'max_PSIS_weight': float(np.max(self.psis_weights)), 'mean_sampled_model_size': float(np.mean(self.G.sum(axis=1)))}

def log_q_eta(G: torch.Tensor, eta_t: torch.Tensor) -> torch.Tensor:
    return (G * eta_t[None, :] - F.softplus(eta_t)[None, :]).sum(dim=1)

def draw_and_score(target: CollapsedLinearTarget, eta: np.ndarray, draw_seed: int, K: int=K_FIXED, batch_size: int=1000, route_audit_draws: int=20) -> DrawResult:
    if int(K) != K_FIXED:
        raise ValueError(f'This comparison fixes K={K_FIXED}; received K={K}.')
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    rho = sigmoid_eta_exact(eta, label='proposal eta')
    if len(eta) != target.p:
        raise ValueError(f'Proposal length {len(eta)} does not match p={target.p}.')
    eta_t = torch.from_numpy(eta).to(dtype=DTYPE, device=DEVICE)
    rho_t = torch.from_numpy(rho).to(dtype=DTYPE, device=DEVICE)
    torch.manual_seed(int(draw_seed))
    G_chunks = []
    lw_chunks = []
    remaining = int(K)
    with torch.no_grad():
        while remaining > 0:
            b = min(int(batch_size), remaining)
            G = torch.bernoulli(rho_t.expand(b, -1))
            logp = target.log_joint(G)
            logq = log_q_eta(G, eta_t)
            lw = logp - logq
            if not bool(torch.isfinite(lw).all().item()):
                raise FloatingPointError('Non-finite raw log importance ratios.')
            G_chunks.append(G.to(dtype=torch.uint8).cpu().numpy())
            lw_chunks.append(lw.cpu().numpy().astype(np.float64))
            remaining -= b
    G_np = np.concatenate(G_chunks, axis=0)
    raw_logw = np.concatenate(lw_chunks, axis=0)
    _, w_raw = _normalise_logweights(raw_logw)
    _, w_psis, khat = psis_smooth_logweights(raw_logw)
    pip = np.asarray(w_psis @ G_np, dtype=np.float64)
    raw_ess = float(1.0 / np.sum(w_raw * w_raw))
    psis_ess = float(1.0 / np.sum(w_psis * w_psis))
    n_audit = min(int(route_audit_draws), len(G_np))
    if n_audit >= 2:
        target.audit_primal_dual(G_np[:n_audit])
    return DrawResult(eta=eta.copy(), rho=rho.copy(), G=G_np, raw_logw=raw_logw, raw_weights=w_raw, psis_weights=w_psis, pip=np.clip(pip, 0.0, 1.0), khat=float(khat), raw_ess=raw_ess, psis_ess=psis_ess, seed=int(draw_seed), K=int(K), psis_status=_PSIS_LAST_STATUS)

def _choose_damped_candidate(candidate: Mapping[float, DrawResult]) -> Tuple[float, str]:
    base = candidate[0.0]
    finite_nonzero = [lam for lam in DAMPED_GRID if lam != 0.0 and np.isfinite(candidate[lam].khat)]
    if np.isfinite(base.khat):
        if not finite_nonzero:
            return (0.0, 'no_finite_nonzero_candidate_k')
        best = min(finite_nonzero, key=lambda lam: (candidate[lam].khat, -candidate[lam].psis_ess, lam))
        if candidate[best].khat <= base.khat - DAMPED_MIN_K_IMPROVEMENT:
            return (float(best), 'validation_k_improved_over_omega0')
        return (0.0, 'no_material_validation_k_improvement')
    if finite_nonzero:
        best = min(finite_nonzero, key=lambda lam: (candidate[lam].khat, -candidate[lam].psis_ess, lam))
        return (float(best), 'baseline_k_nonfinite__finite_damped_candidate')
    return (0.0, 'no_finite_validation_k__keep_original_q')

def run_damped(target: CollapsedLinearTarget, eta_vi: np.ndarray, method: str, run_seed: int, setting_label: str, adapted_prob_eps: float, batch_size: int, ordinary_override: Optional[DrawResult]=None) -> Dict[str, Any]:
    eta_vi = np.asarray(eta_vi, dtype=np.float64).reshape(-1)
    rho_vi = sigmoid_eta_exact(eta_vi, 'original q_VI eta')
    if ordinary_override is None:
        ordinary_seed = stable_seed(run_seed, method, setting_label, '|ORDINARY_SHARED')
        ordinary = draw_and_score(target, eta_vi, ordinary_seed, batch_size=batch_size)
    else:
        ordinary = ordinary_override
        if ordinary.K != K_FIXED:
            raise RuntimeError('ordinary_override must use fixed K=5000.')
        if not np.array_equal(np.asarray(ordinary.eta), eta_vi):
            raise RuntimeError('ordinary_override proposal does not match eta_vi.')
    current_eta = eta_vi.copy()
    current_rho = rho_vi.copy()
    current_k = float(ordinary.khat)
    triggered = bool(not np.isfinite(current_k) or current_k > PARETO_K_STOP)
    success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    stop_reason = 'ordinary_k_at_or_below_0.7' if success else 'adaptation_not_started'
    history = []
    accepted_iterations = 0
    accepted_omegas = []
    first_pilot = None
    first_rho_hat = np.asarray(ordinary.pip, dtype=np.float64)
    max_iters = int(IWMM_MAX_ITERS)
    if triggered:
        for it in range(1, max_iters + 1):
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
            if it == 1:
                pilot_tag = '|DAMPED_PILOT'
                validation_tag = '|DAMPED_VALIDATE'
            else:
                pilot_tag = f'|DAMPED_PILOT|iter={it}'
                validation_tag = f'|DAMPED_VALIDATE|iter={it}'
            pilot_seed = stable_seed(run_seed, method, setting_label, pilot_tag)
            pilot = draw_and_score(target, current_eta, pilot_seed, batch_size=batch_size)
            rho_hat = np.asarray(pilot.pip, dtype=np.float64)
            if it == 1:
                first_pilot = pilot
                first_rho_hat = rho_hat.copy()
            validation_seed = stable_seed(run_seed, method, setting_label, validation_tag)
            validation: Dict[float, DrawResult] = {}
            candidate_eta: Dict[float, np.ndarray] = {}
            candidate_rho: Dict[float, np.ndarray] = {}
            for omega in DAMPED_GRID:
                omega = float(omega)
                raw_rho = (1.0 - omega) * current_rho + omega * rho_hat
                if omega == 0.0:
                    rho = current_rho.copy()
                    eta = current_eta.copy()
                else:
                    rho, eta = adapted_prob_to_eta(raw_rho, adapted_prob_eps, f'damped iteration={it}, omega={omega:g}')
                candidate_rho[omega] = rho
                candidate_eta[omega] = eta
                validation[omega] = draw_and_score(target, eta, validation_seed, batch_size=batch_size)
            selected_omega, reason = _choose_damped_candidate(validation)
            baseline_k = float(validation[0.0].khat)
            # Retain the lowest tested score seen so far. A different
            # validation batch may raise the current-proposal estimate;
            # that does not justify replacing an earlier better candidate.
            selected_k = float(validation[selected_omega].khat)
            improves_best = bool(selected_omega != 0.0
                and np.isfinite(selected_k)
                and (not np.isfinite(current_k)
                     or selected_k < current_k - DAMPED_MIN_K_IMPROVEMENT))
            if selected_omega != 0.0 and not improves_best:
                selected_omega = 0.0
                reason = 'candidate_did_not_reduce_lowest_tested_Pareto_k'
            for omega in DAMPED_GRID:
                omega = float(omega)
                d = validation[omega].diagnostics()
                d.update({'iteration': int(it), 'omega': omega, 'selected': bool(np.isclose(omega, selected_omega) and selected_omega != 0.0), 'baseline_k': baseline_k, 'lowest_tested_k_before': float(current_k), 'selection_reason': reason, 'pilot_seed': int(pilot.seed), 'validation_seed': int(validation[omega].seed), 'mean_abs_probability_change': float(np.mean(np.abs(candidate_rho[omega] - current_rho)))})
                history.append(d)
            if selected_omega == 0.0:
                # Keep the earlier best proposal and its selection score.
                stop_reason = reason
                success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
                break
            current_rho = candidate_rho[selected_omega].copy()
            current_eta = candidate_eta[selected_omega].copy()
            current_k = float(validation[selected_omega].khat)
            accepted_iterations += 1
            accepted_omegas.append(float(selected_omega))
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
        else:
            stop_reason = 'maximum_iterations_reached'
            success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    last_omega = float(accepted_omegas[-1]) if accepted_omegas else 0.0
    if accepted_iterations <= 1:
        final_tag = f'|DAMPED_FINAL|omega={last_omega:.6g}'
    else:
        omega_path = ','.join((f'{x:.6g}' for x in accepted_omegas))
        final_tag = f'|DAMPED_FINAL_ITERATIVE|iters={accepted_iterations}|omegas={omega_path}'
    final_seed = stable_seed(run_seed, method, setting_label, final_tag)
    # Proposal selection uses tested improvement only. The 0.7 cutoff
    # controls when to start/stop the search, never whether an improving
    # proposal is retained. Fresh final draws evaluate that proposal.
    adaptation_used = bool(triggered and accepted_iterations > 0)
    attempted_final = (draw_and_score(target, current_eta, final_seed, batch_size=batch_size)
                       if adaptation_used else None)
    fallback_to_original = bool(triggered and not adaptation_used)
    final = attempted_final if adaptation_used else ordinary
    if fallback_to_original:
        stop_reason += '|kept_original_no_improving_update'
    success = adaptation_used
    return {'method': method, 'run_seed': int(run_seed), 'setting_label': str(setting_label), 'rho_vi': rho_vi, 'eta_vi': eta_vi, 'ordinary': ordinary, 'triggered': triggered, 'pilot': first_pilot, 'pilot_weighted_pip': first_rho_hat, 'selected_omega': last_omega if adaptation_used else 0.0, 'candidate_omega': last_omega, 'selection_reason': stop_reason, 'selected_rho': np.asarray(final.rho).copy(), 'selected_eta': np.asarray(final.eta).copy(), 'candidate_rho': current_rho.copy(), 'candidate_eta': current_eta.copy(), 'final': final, 'history': history, 'adaptation_used': adaptation_used, 'fallback_to_original': fallback_to_original, 'attempted_final_khat': float(attempted_final.khat) if attempted_final is not None else np.nan, 'accepted_iterations': int(accepted_iterations), 'accepted_omegas': tuple(accepted_omegas), 'adaptation_success': bool(success), 'stop_reason': str(stop_reason), 'selection_khat': float(current_k), 'lowest_tested_khat': float(current_k), 'original_khat': float(ordinary.khat), 'final_cutoff_reached': bool(np.isfinite(final.khat) and final.khat <= PARETO_K_STOP), 'final_seed_tag': str(final_tag) if adaptation_used else '|ORDINARY_SHARED', 'candidate_final_seed_tag': str(final_tag) if adaptation_used else None}

def run_iwmm(target: CollapsedLinearTarget, eta_vi: np.ndarray, method: str, run_seed: int, setting_label: str, adapted_prob_eps: float, batch_size: int, ordinary_override: Optional[DrawResult]=None) -> Dict[str, Any]:
    eta_vi = np.asarray(eta_vi, dtype=np.float64).reshape(-1)
    rho_vi = sigmoid_eta_exact(eta_vi, 'original q_VI eta')
    if ordinary_override is None:
        ordinary_seed = stable_seed(run_seed, method, setting_label, '|ORDINARY_SHARED')
        ordinary = draw_and_score(target, eta_vi, ordinary_seed, batch_size=batch_size)
    else:
        ordinary = ordinary_override
        if ordinary.K != K_FIXED:
            raise RuntimeError('ordinary_override must use fixed K=5000.')
        if not np.array_equal(np.asarray(ordinary.eta), eta_vi):
            raise RuntimeError('ordinary_override proposal does not match eta_vi.')
    current_eta = eta_vi.copy()
    current_rho = rho_vi.copy()
    current_eval = ordinary
    current_k = float(ordinary.khat)
    triggered = bool(not np.isfinite(current_k) or current_k > PARETO_K_STOP)
    success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    stop_reason = 'ordinary_k_at_or_below_0.7' if success else 'adaptation_not_started'
    history = []
    accepted = 0
    if triggered:
        for it in range(1, IWMM_MAX_ITERS + 1):
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
            proposed_rho_raw = np.asarray(current_eval.pip, dtype=np.float64)
            proposed_rho, proposed_eta = adapted_prob_to_eta(proposed_rho_raw, adapted_prob_eps, f'IWMM iteration {it}')
            candidate_seed = stable_seed(run_seed, method, setting_label, f'|IWMM_CANDIDATE|iter={it}')
            candidate = draw_and_score(target, proposed_eta, candidate_seed, batch_size=batch_size)
            improves = bool(np.isfinite(candidate.khat) and (not np.isfinite(current_k) or candidate.khat < current_k - 1e-12))
            history.append({'iteration': int(it), 'current_k_before': float(current_k), 'candidate_k': float(candidate.khat), 'candidate_PSIS_ESS': float(candidate.psis_ess), 'accepted': improves, 'candidate_seed': int(candidate.seed), 'mean_abs_probability_change': float(np.mean(np.abs(proposed_rho - current_rho)))})
            if not improves:
                stop_reason = 'candidate_did_not_reduce_Pareto_k'
                break
            current_eta = proposed_eta
            current_rho = proposed_rho
            current_eval = candidate
            current_k = float(candidate.khat)
            accepted += 1
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
        else:
            stop_reason = 'maximum_iterations_reached'
            success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    final_seed = stable_seed(run_seed, method, setting_label, '|IWMM_FINAL_EVAL')
    # Proposal selection uses tested improvement only. The 0.7 cutoff
    # controls when to start/stop the search, never whether an improving
    # proposal is retained. Fresh final draws evaluate that proposal.
    adaptation_used = bool(triggered and accepted > 0)
    attempted_final = (draw_and_score(target, current_eta, final_seed, batch_size=batch_size)
                       if adaptation_used else None)
    fallback_to_original = bool(triggered and not adaptation_used)
    final = attempted_final if adaptation_used else ordinary
    if fallback_to_original:
        stop_reason += '|kept_original_no_improving_update'
    success = adaptation_used
    return {'method': method, 'run_seed': int(run_seed), 'setting_label': str(setting_label), 'rho_vi': rho_vi, 'eta_vi': eta_vi, 'ordinary': ordinary, 'triggered': triggered, 'accepted_iterations': int(accepted), 'adaptation_success': success, 'stop_reason': stop_reason, 'selected_rho': np.asarray(final.rho).copy(), 'selected_eta': np.asarray(final.eta).copy(), 'candidate_rho': current_rho.copy(), 'candidate_eta': current_eta.copy(), 'selected_adaptation_k': float(current_k), 'selection_khat': float(current_k), 'lowest_tested_khat': float(current_k), 'original_khat': float(ordinary.khat), 'final_cutoff_reached': bool(np.isfinite(final.khat) and final.khat <= PARETO_K_STOP), 'final_seed_tag': '|IWMM_FINAL_EVAL' if adaptation_used else '|ORDINARY_SHARED', 'final': final, 'history': history, 'adaptation_used': adaptation_used, 'fallback_to_original': fallback_to_original, 'attempted_final_khat': float(attempted_final.khat) if attempted_final is not None else np.nan}

def _num(x):
    return pd.to_numeric(x, errors='coerce')

def _rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def _ci95(values):
    x = _num(pd.Series(values)).dropna().to_numpy(float)
    n = len(x)
    if n == 0:
        return dict(n=0, mean=np.nan, sd=np.nan, se=np.nan, lo=np.nan, hi=np.nan)
    mean = float(np.mean(x))
    if n == 1:
        return dict(n=1, mean=mean, sd=np.nan, se=np.nan, lo=np.nan, hi=np.nan)
    sd = float(np.std(x, ddof=1))
    se = sd / math.sqrt(n)
    crit = float(student_t.ppf(0.975, n - 1)) if student_t is not None else 1.96
    return dict(n=n, mean=mean, sd=sd, se=se, lo=mean - crit * se, hi=mean + crit * se)

def _eval_elbo_2000(target, eta: np.ndarray, batch_size: int):
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    rho = sigmoid_eta_exact(eta, label='evaluated-ELBO eta')
    eta_t = torch.from_numpy(eta).to(dtype=DTYPE, device=DEVICE)
    rho_t = torch.from_numpy(rho).to(dtype=DTYPE, device=DEVICE)
    torch.manual_seed(int(EVAL_ELBO_SEED))
    vals = []
    active_sum = 0.0
    audit_G = None
    remaining = EVAL_ELBO_SAMPLES
    with torch.no_grad():
        while remaining > 0:
            b = min(int(batch_size), remaining)
            G = torch.bernoulli(rho_t.expand(b, -1))
            if audit_G is None:
                audit_G = G[:min(2, b)].detach().cpu().numpy()
            v = target.log_joint(G) - log_q_eta(G, eta_t)
            if not bool(torch.isfinite(v).all().item()):
                raise FloatingPointError('Non-finite evaluated-ELBO draw.')
            vals.append(v.cpu().numpy().astype(np.float64))
            active_sum += float(G.sum().item())
            remaining -= b
    x = np.concatenate(vals)
    if len(x) != int(EVAL_ELBO_SAMPLES):
        raise RuntimeError('Evaluated-ELBO sample count mismatch.')
    if audit_G is not None and len(audit_G) >= 1:
        target.audit_primal_dual(audit_G, atol=2e-05)
    sd = float(np.std(x, ddof=1))
    return {'evaluated_elbo_2000': float(np.mean(x)), 'evaluated_elbo_2000 MCSE': float(sd / math.sqrt(len(x))), 'evaluated_elbo_2000 sd': sd, 'evaluated_elbo_2000 mean sampled active size': float(active_sum / len(x)), 'evaluated_elbo_2000 target version': EVAL_TARGET_VERSION}

def _select_best_mcse(group):
    g = group.copy()
    g['evaluated_elbo_2000'] = _num(g['evaluated_elbo_2000'])
    g['evaluated_elbo_2000 MCSE'] = _num(g['evaluated_elbo_2000 MCSE'])
    g = g.dropna(subset=['evaluated_elbo_2000', 'evaluated_elbo_2000 MCSE']).sort_values('evaluated_elbo_2000', ascending=False, kind='mergesort').reset_index(drop=True)
    if g.empty:
        raise RuntimeError('No valid evaluated ELBO values in a method/seed group.')
    top = g.iloc[0]
    Lmax = float(top['evaluated_elbo_2000'])
    mmax = float(top['evaluated_elbo_2000 MCSE'])
    gaps = Lmax - _num(g['evaluated_elbo_2000']).to_numpy(float)
    lim = 2.0 * np.sqrt(mmax * mmax + _num(g['evaluated_elbo_2000 MCSE']).to_numpy(float) ** 2)
    tied = g.loc[gaps <= lim].copy()
    tied['_m'] = _num(tied['evaluated_elbo_2000 MCSE'])
    tied['_L'] = _num(tied['evaluated_elbo_2000'])
    tied = tied.sort_values(['_m', '_L'], ascending=[True, False], kind='mergesort')
    out = tied.iloc[0].drop(labels=['_m', '_L']).copy()
    out['raw_highest_elbo'] = Lmax
    out['number_of_mcse_tied_settings'] = int(len(tied))
    out['selection_rule'] = 'all_within_2_combined_mcse_then_smallest_mcse'
    return out

def _posterior_components(G, X, y, Xstar, tau2=1.0, observation_noise=False):
    G = np.asarray(G, np.uint8)
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    Xstar = np.asarray(Xstar, float)
    n, p = X.shape
    nu = float(n - 1)
    xbar = X.mean(0)
    ybar = float(y.mean())
    Xc = X - xbar
    yc = y - ybar
    ycyc = float(yc @ yc)
    loc = np.empty((len(G), len(Xstar)), float)
    scale2 = np.empty_like(loc)
    packed = np.packbits(G, axis=1)
    groups = {}
    for i in range(len(G)):
        groups.setdefault(packed[i].tobytes(), []).append(i)
    for pos in groups.values():
        pos = np.asarray(pos, int)
        idx = np.flatnonzero(G[pos[0]])
        k = len(idx)
        if k == 0:
            m = np.empty(0)
            L = None
            rss = max(ycyc, 1e-08)
        else:
            XA = Xc[:, idx]
            M = XA.T @ XA + (1.0 / float(tau2) + 1e-08) * np.eye(k)
            rhs = XA.T @ yc
            L = np.linalg.cholesky(M)
            m = np.linalg.solve(L.T, np.linalg.solve(L, rhs))
            rss = max(float(ycyc - rhs @ m), 1e-08)
        if k == 0:
            mu = np.full(len(Xstar), ybar)
            lev = np.full(len(Xstar), 1.0 / n)
        else:
            D = Xstar[:, idx] - xbar[idx][None, :]
            mu = ybar + D @ m
            V = np.linalg.solve(L, D.T)
            lev = 1.0 / n + np.sum(V * V, axis=0)
        if observation_noise:
            lev = 1.0 + lev
        loc[pos] = mu[None, :]
        scale2[pos] = rss / nu * lev[None, :]
    return (loc, scale2, nu)

def _interval_score(lo, hi, y, alpha=0.05):
    lo = np.asarray(lo, float)
    hi = np.asarray(hi, float)
    y = np.asarray(y, float)
    score = hi - lo + 2.0 / alpha * (lo - y) * (y < lo) + 2.0 / alpha * (y - hi) * (y > hi)
    return float(np.mean(score))

def _interval_metrics(loc, scale2, weights, df, seed, truth=None):
    w = np.asarray(weights, float)
    w = w / w.sum()
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(w), size=N_MIXTURE_DRAWS, replace=True, p=w)
    draws = loc[idx] + np.sqrt(scale2[idx]) * rng.standard_t(df=float(df), size=(N_MIXTURE_DRAWS, loc.shape[1]))
    lo, hi = np.quantile(draws, [0.025, 0.975], axis=0)
    ans = {'interval_width_95': float(np.mean(hi - lo))}
    if truth is not None:
        ans['coverage_95'] = float(np.mean((truth >= lo) & (truth <= hi)))
        ans['coverage_deviation_95'] = float(abs(ans['coverage_95'] - 0.95))
        ans['interval_score_95'] = _interval_score(lo, hi, truth, alpha=0.05)
    return (ans, lo, hi)

def _ar1_xstar(seed, p):
    rng = np.random.default_rng(880001 + 1009 * int(seed))
    Z = rng.standard_normal((N_XSTAR, p))
    X = np.empty_like(Z)
    X[:, 0] = Z[:, 0]
    sd = math.sqrt(1 - AR1_RHO ** 2)
    for j in range(1, p):
        X[:, j] = AR1_RHO * X[:, j - 1] + sd * Z[:, j]
    return X


def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def digest_object(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False, float_format='%.17g')
    tmp.replace(path)


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False, default=str), encoding='utf-8')
    tmp.replace(path)


def locate_saved(value, directory, subdir=''):
    value = Path(str(value))
    for path in (directory / subdir / value.name, value, ROOT / value):
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f'Saved training file missing: {value} (searched {directory / subdir})')


def assert_target(config, label, check_jitter=False):
    expected_values = [('tau2', TAU2), ('a_pi', A_PI), ('b_pi', B_PI)]
    if check_jitter:
        expected_values.append(('jitter', 1e-8))
    for key, expected in expected_values:
        if key not in config or not np.isclose(float(config[key]), expected, atol=1e-14, rtol=0):
            raise ValueError(f'{label}: wrong {key}; expected {expected}, got {config.get(key)}')


def data_for_seed(seed):
    path = DATA_DIR / f'seed_{seed:02d}.csv'
    if not path.is_file():
        raise FileNotFoundError(f'Expected dataset {path}')
    d = pd.read_csv(path)
    names = [f'X{i}' for i in range(1, P + 1)]
    required = ['Intercept', *names, 'y']
    if not set(required).issubset(d.columns):
        raise ValueError(f'{path}: missing {sorted(set(required) - set(d.columns))[:10]}')
    if not np.allclose(d['Intercept'].to_numpy(float), 1.0):
        raise ValueError(f'{path}: invalid Intercept column')
    X = d[names].to_numpy(dtype=np.float64)
    y = d['y'].to_numpy(dtype=np.float64)
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError(f'{path}: nonfinite observations')
    return X, y, path


def truth_coefficients():
    path = DATA_DIR / 'fixed_beta_and_active_set.csv'
    d = pd.read_csv(path)
    required = {'variable_name', 'is_intercept', 'beta', 'is_true_active'}
    if not required.issubset(d.columns):
        raise ValueError(f'{path}: missing {sorted(required - set(d.columns))}')
    b0 = d.loc[d.is_intercept.astype(int).eq(1), 'beta'].to_numpy(float)
    slopes = d.loc[d.is_intercept.astype(int).eq(0)].set_index('variable_name')
    names = [f'X{i}' for i in range(1, P + 1)]
    if len(b0) != 1 or len(slopes) != P or slopes.index.has_duplicates:
        raise ValueError(f'{path}: expected one intercept and {P} distinct slopes')
    slopes = slopes.loc[names]
    beta = slopes['beta'].to_numpy(float)
    truth = slopes['is_true_active'].to_numpy(float)
    if not np.isfinite(beta).all() or not np.isfinite(b0).all() or not np.isin(truth, (0, 1)).all():
        raise ValueError(f'{path}: invalid true model')
    if not np.array_equal(truth.astype(bool), beta != 0):
        raise ValueError(f'{path}: active indicators disagree with nonzero beta')
    return beta, float(b0[0]), truth.astype(bool), path


def read_q(row, frame=None):
    d = pd.read_csv(row['pip_file']) if frame is None else frame
    if 'variable_name' not in d or d['variable_name'].duplicated().any():
        raise ValueError(f"{row['pip_file']}: duplicate/missing variable names")
    if len(d) != P:
        raise ValueError(f"{row['pip_file']}: expected {P} predictors, got {len(d)}")
    if d['variable_name'].tolist() != [f'X{i}' for i in range(1, P + 1)]:
        d = d.set_index('variable_name').loc[[f'X{i}' for i in range(1, P + 1)]].reset_index()
    eta = d[row['lambda_column']].to_numpy(dtype=float, copy=True)
    pip = d[row['pip_column']].to_numpy(dtype=float, copy=True)
    if not np.isfinite(eta).all() or not np.isfinite(pip).all() or np.any((pip < 0) | (pip > 1)):
        raise ValueError(f"{row['pip_file']}: invalid saved logits/PIPs")
    if np.max(np.abs(expit(eta) - pip)) > 1e-7:
        raise ValueError(f"{row['pip_file']}: saved PIPs disagree with logits")
    return pip, eta


def expected_grid(method):
    lrs = (.01, .05, .1, .2, .3)
    if method == 'ELBO':
        return {(lr, 'ELBO', 300, 1) for lr in lrs}
    if 'annealed' in method:
        return {(lr, 'annealed', 1, 300) for lr in lrs}
    return {(lr, alpha, 1, 300) for lr in lrs for alpha in (.2, .5, .7, .9)}



def select_best_mcse(group):
    return _select_best_mcse(pd.DataFrame(group))

def eval_one(row, target):
    _, eta = read_q(row)
    evaluated = _eval_elbo_2000(target, eta, batch_size=250)
    return {'config_id':row['config_id'], **evaluated}

def score_one(row, target, beta, beta0, truth, weights_dir):
    pip_vi, eta = read_q(row)
    seed, method = int(row['seed']), str(row['method'])
    label = str(row['pip_column'])
    ordinary = draw_and_score(target, eta, stable_seed(seed, method, label, '|ORDINARY_SHARED'), batch_size=BATCH_SIZE)
    damp = run_damped(target, eta, method, seed, label, ADAPTED_PROB_EPS, BATCH_SIZE, ordinary_override=ordinary)
    iw = run_iwmm(target, eta, method, seed, label, ADAPTED_PROB_EPS, BATCH_SIZE, ordinary_override=ordinary)
    stages = {
        'Original VI': (pip_vi, ordinary.G, np.full(K_FIXED, 1/K_FIXED), np.nan, ordinary),
        'Original PSIS': (ordinary.pip, ordinary.G, ordinary.psis_weights, ordinary.khat, ordinary),
        'Damped PSIS': (damp['final'].pip, damp['final'].G, damp['final'].psis_weights, damp['final'].khat, damp['final']),
        'IWMM PSIS': (iw['final'].pip, iw['final'].G, iw['final'].psis_weights, iw['final'].khat, iw['final'])
    }
    X, y = target.X, target.y
    Xstar = _ar1_xstar(seed, P)
    ftrue = beta0 + Xstar @ beta
    records, pips, component_cache = [], {}, {}
    for stage, (pip, G, weights, khat, draw) in stages.items():
        key = id(G)
        if key not in component_cache:
            component_cache[key] = _posterior_components(G, X, y, Xstar, tau2=TAU2, observation_noise=False)
        loc, scale2, nu = component_cache[key]
        im, _, _ = _interval_metrics(loc, scale2, weights, nu, 990001 + seed, truth=ftrue)
        chosen = pip > 0.5
        tp = int(np.sum(chosen & truth))
        fp = int(np.sum(chosen & ~truth))
        fn = int(np.sum(~chosen & truth))
        record = {k:row[k] for k in ('config_id','method','seed','setting','learning_rate','alpha_setting','S','N')}
        record.update(stage=stage, psis_status=('not_applicable' if stage == 'Original VI' else draw.psis_status), **{'PIP RMSE vs truth':_rmse(pip, truth.astype(float)),
                   'F1 vs truth':float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 1.,
                   'Mean marginal FKL vs truth':forward_kl_product_bernoulli(truth.astype(float), pip)/P,
                   'Pareto k-hat':float(khat), '95% latent coverage':im['coverage_95'],
                   '95% interval score':im['interval_score_95'],
                   'training_time_seconds':float(row['training_time_seconds']),
                   'training_iterations':int(row['iters']),
                   'final_alpha':row['final_alpha'],
                   'importance_ESS':float(draw.psis_ess) if stage != 'Original VI' else np.nan})
        if '_reference_pip' in row:
            ref = np.asarray(row['_reference_pip'])
            record['PIP RMSE vs Gibbs'] = _rmse(pip, ref)
            record['Mean excess marginal FKL vs Gibbs'] = forward_kl_product_bernoulli(ref, pip)/P
        records.append(record)
        pips[stage] = np.asarray(pip, dtype=np.float64).copy()
    for stage, (_, _, _, _, draw) in stages.items():
        if stage == 'Original VI':
            continue
        folder = weights_dir; folder.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(folder / (str(row['config_id']) + '_' + stage.replace(' ', '_') + '.npz'),
            raw_logw=draw.raw_logw, psis_weights=draw.psis_weights, raw_weights=draw.raw_weights,
            khat=draw.khat, seed=draw.seed, status=draw.psis_status)
    result = {'analysis_version':ANALYSIS_VERSION, 'config_id':row['config_id'],
              'records':records, 'pips':pips,
              'diagnostics':{'original':ordinary.diagnostics(),
                             'damped':{'selection_khat':damp['selection_khat'], 'original_khat':damp['original_khat'], 'triggered':damp['triggered'], 'accepted_iterations':damp['accepted_iterations'],
                                       'accepted_omegas':damp['accepted_omegas'], 'stop_reason':damp['stop_reason'],
                                       'final':damp['final'].diagnostics(), 'history':damp['history']},
                             'iwmm':{'selection_khat':iw['selection_khat'], 'original_khat':iw['original_khat'], 'triggered':iw['triggered'], 'accepted_iterations':iw['accepted_iterations'],
                                     'stop_reason':iw['stop_reason'], 'final':iw['final'].diagnostics(), 'history':iw['history']}}}
    return result

def summary_table(data, keys):
    rows = []
    for group, g in data.groupby(keys, sort=False, dropna=False):
        key = group if isinstance(group, tuple) else (group,)
        row = dict(zip(keys, key))
        for metric in METRICS:
            x = pd.to_numeric(g[metric], errors='coerce').to_numpy(float)
            finite = x[np.isfinite(x)]
            row[metric] = float(finite.mean()) if len(finite) else np.nan
            row[metric + ' SD across replicates'] = float(finite.std(ddof=1)) if len(finite) > 1 else np.nan
            row[metric + ' n_finite'] = len(finite)
        row['n_replicates'] = len(g)
        rows.append(row)
    return pd.DataFrame(rows)

def export_table(frame, name, out, caption):
    atomic_csv(frame, out / f'{name}.csv')
    view = frame.copy()
    for metric in METRICS:
        sd = metric + ' SD across replicates'
        if sd in view:
            view[metric] = [f'{m:.4f} ({s:.4f})' if np.isfinite(m) and np.isfinite(s)
                            else f'{m:.4f} (--)' if np.isfinite(m) else '--'
                            for m,s in zip(view[metric],view[sd])]
            view = view.drop(columns=[sd, metric + ' n_finite'], errors='ignore')
    headings={'PIP RMSE vs truth':'RMSE','F1 vs truth':r'$F_1$',
              'Mean marginal FKL vs truth':'Mean FKL','Pareto k-hat':r'$\widehat{k}$',
              '95% latent coverage':'Coverage','95% interval score':'Interval score'}
    def escaped(value):
        return str(value).replace('&',r'\&').replace('%',r'\%').replace('_',r'\_').replace('#',r'\#')
    def cell(value):
        if pd.isna(value):
            return '--'
        if isinstance(value,(float,np.floating)):
            return f'{value:.4f}' if np.isfinite(value) else '--'
        return escaped(value)
    is_long=len(view)>35
    columns='l'*len(view.columns)
    begin=r'\begin{longtable}{'+columns+'}' if is_long else r'\begin{tabular}{'+columns+'}'
    end=r'\end{longtable}' if is_long else r'\end{tabular}'
    lines=[r'\small',begin]
    if is_long:
        lines.append(r'\caption{'+escaped(caption)+r'}\label{tab:'+escaped(name.lower())+r'}\\')
    lines += [r'\toprule',' & '.join(headings.get(col,escaped(col)) for col in view.columns)+r' \\',r'\midrule']
    for _, record in view.iterrows():
        lines.append(' & '.join(cell(record[col]) for col in view.columns)+r' \\')
    lines += [r'\bottomrule',end]
    if not is_long:
        lines=[r'\begin{table}[tbp]',r'\centering',*lines,
               r'\caption{'+escaped(caption)+r'}',r'\label{tab:'+escaped(name.lower())+r'}',r'\end{table}']
    (out / f'{name}.tex').write_text('\n'.join(lines)+'\n', encoding='utf-8')

def paired_statistics(main):
    rows = []
    for stage in STAGES:
        base = main[(main.method == 'ELBO') & (main.stage == stage)].set_index('seed')
        for method in METHODS[1:]:
            cur = main[(main.method == method) & (main.stage == stage)].set_index('seed')
            if set(cur.index) != set(base.index) or cur.index.has_duplicates or base.index.has_duplicates:
                raise ValueError(f'{method}/{stage}: unpaired dataset seeds')
            for metric in METRICS:
                raw = cur[metric] - base[metric]
                oriented = ((cur[metric]-0.95).abs()-(base[metric]-0.95).abs()
                            if metric == '95% latent coverage' else -raw if metric == 'F1 vs truth' else raw)
                oriented_ci, raw_ci = _ci95(oriented), _ci95(raw)
                rows.append({'stage':stage,'method':method,'metric':metric,'n':oriented_ci['n'],
                             'mean_paired_difference':raw_ci['mean'], 'sd_paired_difference':raw_ci['sd'],
                             '95% interval lower':raw_ci['lo'], '95% interval upper':raw_ci['hi'],
                             'oriented_mean_difference_left_is_better':oriented_ci['mean'],
                             'oriented_95% interval lower':oriented_ci['lo'],
                             'oriented_95% interval upper':oriented_ci['hi']})
    return pd.DataFrame(rows)

def _plot_panel(ax,data,metric,methods,stages,styles,title):
    stage_offset,stage_marker,stage_color=styles
    for stage in stages:
        dd=data[(data.metric==metric)&(data.stage==stage)].set_index('method').reindex(methods)
        x=pd.to_numeric(dd['oriented_mean_difference_left_is_better'],errors='coerce').to_numpy(float)
        lo=pd.to_numeric(dd['oriented_95% interval lower'],errors='coerce').to_numpy(float)
        hi=pd.to_numeric(dd['oriented_95% interval upper'],errors='coerce').to_numpy(float)
        good=np.isfinite(x)&np.isfinite(lo)&np.isfinite(hi)
        y=np.arange(len(methods))+stage_offset[stage]
        if np.any(good):
            ax.errorbar(x[good],y[good],xerr=[x[good]-lo[good],hi[good]-x[good]],
                        fmt=stage_marker[stage],color=stage_color[stage],capsize=3,
                        linewidth=0,elinewidth=1.1,markersize=5.3,label=stage)
    ax.axvline(0,color='0.35',linewidth=1,linestyle='--')
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods)
    ax.invert_yaxis()
    ax.set_title(title,loc='left',fontsize=10)
    ax.set_xlabel('Paired difference vs ELBO')
    ax.grid(axis='x',alpha=.22)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))

def paired_five_panels(paired,out):
    methods=METHODS[1:]
    stages=['Original VI','Original PSIS','IWMM PSIS','Damped PSIS']
    metrics=['PIP RMSE vs Gibbs','Mean excess marginal FKL vs Gibbs','PIP RMSE vs truth','F1 vs truth','95% interval score']
    titles=[r'(a) $\Delta$ PIP RMSE vs Gibbs',r'(b) $\Delta$ Mean excess FKL vs Gibbs',r'(c) $\Delta$ PIP RMSE vs truth',r'(d) $-\Delta F_1$ vs truth',r'(e) $\Delta$ 95% interval score']
    colors=plt.rcParams['axes.prop_cycle'].by_key()['color']
    styles=({'Original VI':-.27,'Original PSIS':-.09,'IWMM PSIS':.09,'Damped PSIS':.27},
            {'Original VI':'o','Original PSIS':'s','IWMM PSIS':'^','Damped PSIS':'D'},
            {stage:colors[i] for i,stage in enumerate(stages)})
    fig=plt.figure(figsize=(16.2,8.8))
    grid=fig.add_gridspec(2,6,hspace=.55,wspace=.55)
    axes=[fig.add_subplot(grid[0,:3]),fig.add_subplot(grid[0,3:]),
          fig.add_subplot(grid[1,:2]),fig.add_subplot(grid[1,2:4]),fig.add_subplot(grid[1,4:])]
    for ax,metric,title in zip(axes,metrics,titles):
        _plot_panel(ax,paired,metric,methods,stages,styles,title)
    for ax in (axes[1],axes[3],axes[4]):
        ax.tick_params(axis='y',labelleft=False)
    handles={}
    for ax in axes:
        hs,ls=ax.get_legend_handles_labels()
        handles.update(zip(ls,hs))
    stage_labels={'Damped PSIS':'Iterative Damped PSIS'}
    present=[s for s in stages if s in handles]
    if present:
        fig.legend([handles[s] for s in present],[stage_labels.get(s,s) for s in present],
                   loc='lower center',ncol=len(present),frameon=False,bbox_to_anchor=(.5,.02))
    fig.suptitle('VIMCO methods versus ELBO at p = 1000',fontsize=15,y=.98)
    fig.subplots_adjust(left=.105,right=.985,bottom=.16,top=.89)
    plot_dir=out/'plots';plot_dir.mkdir(exist_ok=True)
    for suffix in ('png','pdf'):
        fig.savefig(plot_dir/f'fig_p1000_paired_five_panels.{suffix}',dpi=300,bbox_inches='tight')
    plt.close(fig)

def scatter_plots(pip_index,truth,out):
    plot_dir=out/'plots';plot_dir.mkdir(exist_ok=True)
    rng=np.random.default_rng(271828)
    columns={stage:f'pip_{stage.lower().replace(" ","_")}' for stage in STAGES}
    sampled=[]
    for method in METHODS:
        files=pip_index[pip_index.method.eq(method)].sort_values('seed').pip_file.tolist()
        # All active predictors and a reproducible sample of inactive predictors are plotted.
        pieces=[]
        for file in files:
            table=pd.read_csv(file)
            active=np.flatnonzero(table.is_true_active.to_numpy(int))
            inactive=np.flatnonzero(1-table.is_true_active.to_numpy(int))
            take=rng.choice(inactive,size=min(300,len(inactive)),replace=False)
            pieces.append(table.iloc[np.concatenate((active,take))][['is_true_active',*columns.values()]])
        dd=pd.concat(pieces,ignore_index=True)
        dd['method']=method
        sampled.append(dd)
    scatter=pd.concat(sampled,ignore_index=True)
    fig,axes=plt.subplots(2,3,figsize=(15,9),squeeze=False)
    for ax,method in zip(axes.flat,METHODS):
        d=scatter[scatter.method.eq(method)]
        x=d.is_true_active.to_numpy(float)+rng.normal(0,.035,len(d))
        ax.scatter(x,d[columns['Original VI']],s=2.3,alpha=.20,rasterized=True)
        ax.set(xlim=(-.13,1.13),ylim=(-.03,1.03),xticks=(0,1),xlabel='True inclusion',ylabel='Estimated PIP',title=method)
        ax.grid(alpha=.2)
    axes.flat[-1].axis('off')
    fig.suptitle('Selected configuration: original VI PIPs and known active set')
    fig.tight_layout()
    for suffix in ('png','pdf'):
        fig.savefig(plot_dir/f'fig_p1000_truth_pip_by_method.{suffix}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    focus=scatter[scatter.method.eq('VIMCO-GM annealed')]
    fig,axes=plt.subplots(2,2,figsize=(11,9))
    for ax,stage in zip(axes.flat,STAGES):
        x=focus.is_true_active.to_numpy(float)+rng.normal(0,.035,len(focus))
        ax.scatter(x,focus[columns[stage]],s=2.3,alpha=.20,rasterized=True)
        ax.set(xlim=(-.13,1.13),ylim=(-.03,1.03),xticks=(0,1),xlabel='True inclusion',ylabel='Estimated PIP',title='Iterative Damped PSIS' if stage=='Damped PSIS' else stage)
        ax.grid(alpha=.2)
    fig.suptitle('Annealed VIMCO-GM: selected PIPs across processing stages')
    fig.tight_layout()
    for suffix in ('png','pdf'):
        fig.savefig(plot_dir/f'fig_p1000_truth_pip_by_stage.{suffix}',dpi=300,bbox_inches='tight')
    plt.close(fig)


# Execution, provenance and output files.
import importlib.metadata
import inspect
import platform


def backend_provenance():
    versions = {}
    for package in ('numpy','scipy','torch','arviz','arviz-stats','pandas'):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    function = az.psislw if az is not None and hasattr(az, 'psislw') else _arviz_array_stats.psislw if _arviz_array_stats is not None else None
    try:
        source = inspect.getsource(function)
    except (OSError, TypeError):
        source = repr(type(function))
    return dict(versions=versions, python=platform.python_version(),
                backend='arviz.psislw' if az is not None and hasattr(az,'psislw') else 'arviz_stats.array_stats.psislw',
                psis_source_sha256=hashlib.sha256(source.encode()).hexdigest())


def metadata_path(directory, method, seed):
    split = 'S_300_N_1' if method == 'ELBO' else 'S_1_N_300'
    named = directory / f'runtime_and_metrics_{safe_method_name(method)}_{split}_seed_{seed:02d}.csv'
    return named if named.exists() else directory / f'runtime_and_metrics_seed_{seed:02d}.csv'


def canonical_grid_value(value, allowed, label):
    """Match CSV round-trip noise only; do not accept a different tuning grid."""
    number = float(value)
    matches = [float(v) for v in allowed
               if np.isclose(number, float(v), rtol=0.0, atol=1e-12)]
    if len(matches) != 1:
        raise ValueError(f'{label}: {value!r} is not in the prescribed grid {allowed}')
    return matches[0]

In [ ]:
def load_inputs():
    beta, beta0, truth, truth_path = truth_coefficients()
    truth_hash = digest_file(truth_path)
    rows, skipped = [], []
    for seed in EXPECTED_SEEDS:
        files = {m:metadata_path(d,m,seed) for m,d in TRAINING_DIRS.items()}
        if not all(p.is_file() for p in files.values()) or not (DATA_DIR/f'seed_{seed:02d}.csv').is_file():
            missing = [str(p) for p in files.values() if not p.is_file()]
            if not (DATA_DIR/f'seed_{seed:02d}.csv').is_file():
                missing.append(str(DATA_DIR/f'seed_{seed:02d}.csv'))
            skipped.append(dict(seed=seed,reason='missing data or method metadata',details=json.dumps(missing))); continue
        data_hash=digest_file(DATA_DIR/f'seed_{seed:02d}.csv')
        seed_rows=[]; incomplete=False; grid_gaps=[]
        for method, path in files.items():
            frame=pd.read_csv(path)
            found=set()
            for _,r in frame.iterrows():
                s,n=int(r.S),int(r.N)
                if (s,n) != ((300,1) if method=='ELBO' else (1,300)):continue
                assert_target(r,str(path))
                if int(r.seed)!=seed or int(r.method_seed)!=12345:
                    raise ValueError(f'{path}: wrong seed')
                alpha='ELBO' if method=='ELBO' else 'annealed' if 'annealed' in method else canonical_grid_value(r.alpha_setting, (.2,.5,.7,.9), str(path)+' alpha')
                lr = canonical_grid_value(r.learning_rate, (.01,.05,.1,.2,.3), str(path)+' learning rate')
                setting=(lr,alpha,s,n)
                if setting in found:raise ValueError(f'{path}: duplicate setting {setting}')
                found.add(setting)
                qpath=locate_saved(r.pip_file,TRAINING_DIRS[method])
                row=r.to_dict(); row.update(method=method,seed=seed,S=s,N=n,learning_rate=lr,alpha_setting=alpha,setting=json.dumps(setting),pip_file=str(qpath),training_time_seconds=float(r.get('time (s)',np.nan)),final_alpha=r.get('final_alpha',np.nan))
                pip,eta=read_q(row)
                if str(r.get('dataset_sha256'))!=data_hash or str(r.get('true_model_sha256'))!=truth_hash:raise ValueError(f'{path}: data hashes mismatch')
                expected={'VIMCO* annealed':'Annealed VIMCO*','VIMCO-GM annealed':'Annealed VIMCO-GM'}.get(method,method)
                if str(r.method)!=expected:raise ValueError(f'{path}: method mismatch')
                if method!='ELBO':
                    gradient='vimco_star_log_loo_v1' if '*' in method else 'vimco_gm_log_loo_v1'
                    if r.get('gradient_implementation')!=gradient:raise ValueError(f'{path}: unknown gradient implementation')
                row['training_source']='training_notebook'
                row['proposal_hash']=hashlib.sha256(eta.astype('<f8').tobytes()).hexdigest()
                row['data_hash']=data_hash;row['truth_hash']=truth_hash
                seed_rows.append(row)
            if found != expected_grid(method):
                incomplete=True
                grid_gaps.append(dict(method=method, metadata_file=str(path),
                    missing=sorted(expected_grid(method)-found,key=str),
                    extra=sorted(found-expected_grid(method),key=str)))
        if incomplete:
            skipped.append(dict(seed=seed,reason='incomplete S=1 tuning grid',details=json.dumps(grid_gaps)));continue
        rows.extend(seed_rows)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    report = OUTPUT_ROOT / 'input_validation_report.csv'
    atomic_csv(pd.DataFrame(skipped,columns=['seed','reason','details']), report)
    if not rows:
        examples = '\n'.join(f"seed {r['seed']}: {r['reason']} — {r['details']}" for r in skipped[:3])
        raise ValueError(f'No complete shared seeds across all five methods.\nInput report: {report}\n{examples}')
    print('Complete shared dataset seeds:', sorted({r['seed'] for r in rows}), flush=True)
    return pd.DataFrame(rows),beta,beta0,truth,pd.DataFrame(skipped)


def load_gibbs(seeds):
    refs={};provenance=[]
    folder=ROOT/'output_gibbs_tau2_1'
    for seed in seeds:
        meta=folder/f'runtime_and_metrics_gibbs_seed_{seed:02d}.csv'
        table=pd.read_csv(meta)
        if len(table)!=1:raise ValueError(f'{meta}: expected one metadata row')
        row=table.iloc[0];assert_target(row,str(meta))
        data_hash=digest_file(DATA_DIR/f'seed_{seed:02d}.csv');truth_hash=digest_file(DATA_DIR/'fixed_beta_and_active_set.csv')
        chain_pips=[];hashes=[]
        if row.data_sha256!=data_hash or row.true_model_sha256!=truth_hash:raise ValueError(f'{meta}: data mismatch')
        for chain in range(1,int(row.n_chains)+1):
            cp=folder/f'gibbs_draws_seed_{seed:02d}_chain_{chain}.npz'
            with np.load(cp) as saved:draws=saved['samples']
            if draws.shape!=(int(row.retained_sweeps_per_chain),P) or not np.isin(draws,[0,1]).all():raise ValueError(f'{cp}: invalid draws')
            chain_pips.append(draws.mean(0));hashes.append(digest_file(cp))
        if len(chain_pips)!=int(row.n_chains):raise ValueError(f'{meta}: missing chains')
        refs[seed]=np.mean(chain_pips,axis=0);provenance.append(dict(seed=seed,hashes=hashes))
    return refs,provenance


def score_task(row,beta,beta0,truth,weights_dir):
    X,y,_=data_for_seed(int(row['seed']))
    target=CollapsedLinearTarget(X,y,TargetSpec(TAU2,A_PI,B_PI,primal_active_max=80,primal_batch_size=32))
    evaluated=eval_one(row,target)
    scored=score_one(row,target,beta,beta0,truth,weights_dir)
    return {**row,**evaluated},scored


def run_postprocessing():
    torch.set_num_threads(TORCH_THREADS)
    validate_psis_backend_sign()
    candidates,beta,beta0,truth,skipped=load_inputs()
    seeds=sorted(candidates.seed.unique().tolist())
    refs,gibbs_provenance=load_gibbs(seeds)
    provenance=backend_provenance()
    signature=digest_object(dict(source=ANALYSIS_VERSION,environment=provenance,
        settings=[TAU2,A_PI,B_PI,K_FIXED,EVAL_ELBO_SAMPLES,EVAL_ELBO_SEED,PSIS_BASE_SEED,N_XSTAR,N_MIXTURE_DRAWS,BATCH_SIZE,ADAPTED_PROB_EPS,IWMM_MAX_ITERS,DAMPED_GRID,PARETO_K_STOP],gibbs=gibbs_provenance,
        candidates=candidates[['method','seed','setting','proposal_hash','data_hash','truth_hash','training_source']].to_dict('records')))
    out=OUTPUT_ROOT/signature[:16];out.mkdir(parents=True,exist_ok=True)
    weights_dir=out/'weight_arrays'
    candidates['config_id']=[digest_object([signature,r.method,int(r.seed),r.setting,r.proposal_hash])[:24] for r in candidates.itertuples()]
    atomic_csv(skipped,out/'skipped_seeds.csv')
    rows=candidates.to_dict('records')
    evaluated=[];records=[];results={}
    for i,row in enumerate(rows,1):
        if row['seed'] in refs:row['_reference_pip']=refs[row['seed']]
        ev,score=score_task(row,beta,beta0,truth,weights_dir)
        ev.pop('_reference_pip',None);evaluated.append(ev);records.extend(score['records']);results[ev['config_id']]=score
        print(f'Completed {i}/{len(rows)} configurations',flush=True)
    evaluated=pd.DataFrame(evaluated);all_metrics=pd.DataFrame(records)
    selected=pd.DataFrame([select_best_mcse(g) for _,g in evaluated.groupby(['seed','method'],sort=False)])
    main=all_metrics[all_metrics.config_id.isin(selected.config_id)].copy()
    atomic_csv(evaluated,out/'all_configurations_evaluated.csv');atomic_csv(selected,out/'selected_configurations.csv')
    atomic_csv(all_metrics,out/'all_configuration_metrics.csv');atomic_csv(main,out/'selected_configuration_metrics.csv')
    export_table(summary_table(main,['method','stage']),'MAIN_selected_S1_metrics',out,'Selected configurations; means and SD across datasets. Finite Pareto diagnostics only; see availability counts.')
    finite_metrics=all_metrics.copy()
    finite_metrics[METRICS]=finite_metrics[METRICS].replace([np.inf,-np.inf],np.nan)
    per_seed=finite_metrics.groupby(['seed','method','stage'],as_index=False)[METRICS].mean()
    counts=all_metrics.groupby(['seed','method','stage']).size().rename('configurations_per_replicate').reset_index()
    atomic_csv(per_seed.merge(counts),out/'configuration_averages_by_seed.csv')
    export_table(summary_table(per_seed,['method','stage']),'MAIN_configuration_averaged_S1',out,'Average within each dataset over tuning configurations, then summarise across datasets.')
    sd=finite_metrics.groupby(['seed','method','stage'],as_index=False)[METRICS].std(ddof=1)
    atomic_csv(sd,out/'configuration_SD_by_seed.csv')
    export_table(summary_table(sd,['method','stage']),'APPENDIX_configuration_SD',out,'Within-dataset SD across tuning configurations; summarised across datasets.')
    lr=finite_metrics.groupby(['seed','method','stage','learning_rate'],as_index=False)[METRICS].mean()
    export_table(summary_table(lr,['method','stage','learning_rate']),'APPENDIX_learning_rate_sensitivity',out,'Learning-rate sensitivity, averaging other tuning settings within each dataset first.')
    runtime=selected.groupby('method').training_time_seconds.agg(['mean','std','count']).reset_index()
    export_table(runtime,'MAIN_selected_training_runtime',out,'Training time for the selected fit only; excludes grid-search and post-processing costs.')
    paired=paired_statistics(main);atomic_csv(paired,out/'paired_comparisons_vs_ELBO.csv');paired_five_panels(paired,out)
    indices=[];diagnostics=[]
    (out/'selected_pips').mkdir(exist_ok=True)
    for row in selected.to_dict('records'):
        result=results[row['config_id']];file=out/'selected_pips'/f"seed_{int(row['seed']):02d}_{safe_method_name(row['method'])}.csv"
        data={'variable_name':[f'X{i}' for i in range(1,P+1)],'is_true_active':truth.astype(int),'beta':beta}
        data.update({f"pip_{s.lower().replace(' ','_')}":result['pips'][s] for s in STAGES})
        atomic_csv(pd.DataFrame(data),file);indices.append(dict(seed=row['seed'],method=row['method'],pip_file=str(file)))
    scatter_plots(pd.DataFrame(indices),truth,out)
    for config,result in results.items():
        for stage,key in [('Original PSIS','original'),('Damped PSIS','damped'),('IWMM PSIS','iwmm')]:
            d=result['diagnostics'][key]
            diagnostics.append(dict(config_id=config,stage=stage,**(d if key=='original' else d['final'])))
    atomic_csv(pd.DataFrame(diagnostics),out/'weight_diagnostics.csv')
    availability=all_metrics.groupby(['method','stage','psis_status']).size().rename('count').reset_index()
    atomic_csv(availability,out/'Pareto_diagnostic_availability.csv')
    (out/'adaptation_diagnostics.json').write_text(json.dumps({k:v['diagnostics'] for k,v in results.items()},default=str,allow_nan=True))
    manifest=dict(analysis_signature=signature,environment=provenance,seeds=seeds,tau2=TAU2,
        n_configurations=len(candidates),n_selected=len(selected),K=K_FIXED,PSIS_BASE_SEED=PSIS_BASE_SEED,
        selection_rule='evaluated ELBO with 2 combined MCSE tie rule',
        unavailable_k='NaN, explicit status; equal-tail exception uses raw normalised weights',
        adaptation='retain improving lowest tested k; fresh final evaluation reported separately',
        training_sources=candidates.training_source.value_counts().to_dict())
    (out/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
    (OUTPUT_ROOT/'LATEST_RESULTS.txt').write_text(str(out)+'\n')
    print('Saved results:',out)
    return out

In [ ]:
RESULTS_DIR = run_postprocessing()

## Thesis tables and figures

Built only from the saved post-processing results above; nothing is retrained or re-weighted. Outputs go to `<results folder>/THESIS_TABLES_AND_FIGURES`: the main results table with Pareto $\hat k$ and ESS (`.csv` and `.tex`), selected $\alpha$, iterations and runtime, the learning-rate summary, the Original VI configuration-sensitivity summary, and the paired and PIP figures.

In [ ]:
"""Thesis tables and figures, built only from the saved post-processing results.

make_thesis_outputs(setting, results_dir) writes everything into
results_dir / 'THESIS_TABLES_AND_FIGURES'. Nothing is retrained or re-weighted here.

Main results table (Tables 4.3-4.5, 5.1, 5.2) with the Pareto k-hat and ESS columns
Selected fixed alpha and final annealed alpha (Table 5.4)
Mean iterations to termination and selected-fit runtime (Tables 5.5, 5.3)
Learning rate maximising configuration-averaged evaluated ELBO (Table C.1)
Original VI configuration averages and within-replicate configuration SD (Table G.1)
Paired comparisons with ELBO (Figures 4.3, 4.5, 4.6, 5.1, 5.2) and PIP scatter plots
p = 30 only: evaluated-ELBO allocation F-tests (Table B.1) and the varbvs benchmark row
"""
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
from scipy.stats import f as f_distribution

TABLE_METHODS = ['ELBO', 'VIMCO*', 'VIMCO* annealed', 'VIMCO-GM', 'VIMCO-GM annealed']
METHOD_TEX = {'ELBO': 'ELBO', 'VIMCO*': r'\vstar', 'VIMCO* annealed': r'\avstar',
              'VIMCO-GM': r'\vgm', 'VIMCO-GM annealed': r'\avgm'}
TABLE_STAGES = ['Original VI', 'Original PSIS', 'IWMM PSIS', 'Damped PSIS']
STAGE_TEX = {'Original VI': 'Original VI', 'Original PSIS': 'Ordinary PSIS', 'IWMM PSIS': 'IWMM',
             'Damped PSIS': r'\shortstack[l]{Iterative Damped\\PSIS}'}
K_COL, ESS_COL = 'Pareto k-hat', 'importance_ESS'

# Columns of each results table: (metric column, LaTeX header, decimals).
# vi_k: the Original VI row shows the k-hat of the VI proposal (= Ordinary PSIS k-hat).
SETTINGS = {
    'p30': dict(
        label='p30', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs Gibbs', 'PIP RMSE', 4), ('Mean excess marginal FKL vs Gibbs', r'$\Delta\mathrm{FKL}$', 4),
              (K_COL, r'$\widehat k$', 3), (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4),
              ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs Gibbs', 'PIP RMSE'), ('Mean excess marginal FKL vs Gibbs', 'Excess marginal KL'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics_S1.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_settings_S1_N300.csv',
        paired=[('PIP RMSE vs Gibbs', 'Δ PIP RMSE vs Gibbs'), ('Mean excess marginal FKL vs Gibbs', 'Δ Mean marginal KL vs Gibbs'),
                ('95% interval score', 'Δ 95% interval score')]),
    'p1000': dict(
        label='p1000', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs Gibbs', 'PIP RMSE', 4), ('Mean excess marginal FKL vs Gibbs', r'$\Delta\mathrm{FKL}$', 4),
              ('PIP RMSE vs truth', 'PIP RMSE', 4), ('F1 vs truth', '$F_1$', 4), (K_COL, r'$\widehat k$', 3),
              (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs Gibbs', 'PIP RMSE'), ('Mean excess marginal FKL vs Gibbs', 'Excess marginal KL'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='all_configurations_evaluated.csv',
        selected_training='selected_configurations.csv',
        paired=[('PIP RMSE vs Gibbs', 'Δ PIP RMSE vs Gibbs'), ('Mean excess marginal FKL vs Gibbs', 'Δ Mean marginal KL vs Gibbs'),
                ('PIP RMSE vs truth', 'Δ PIP RMSE vs truth'), ('F1 vs truth', '−Δ F1 vs truth'),
                ('95% interval score', 'Δ 95% interval score')]),
    'p5000': dict(
        label='p5000', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs truth', 'PIP RMSE', 4), ('F1 vs truth', '$F_1$', 4), (K_COL, r'$\widehat k$', 3),
              (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs truth', 'PIP RMSE'), ('F1 vs truth', '$F_1$'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='all_configurations_evaluated.csv',
        selected_training='selected_configurations.csv',
        paired=[('PIP RMSE vs truth', 'Δ PIP RMSE vs truth'), ('F1 vs truth', '−Δ F1 vs truth'),
                ('95% interval score', 'Δ 95% interval score')]),
    'bodyfat': dict(
        label='bodyfat', replicates='optimisation seeds', vi_k=False,
        cols=[('PIP RMSE vs exact', 'PIP RMSE', 4), ('Mean excess marginal FKL vs exact', r'$\Delta\mathrm{FKL}$', 4),
              (K_COL, r'Mean $\widehat k$', 3), (ESS_COL, 'Mean ESS', 0),
              ('Exact posterior mass in 95% latent interval', 'Coverage', 4),
              ('Exact-posterior expected 95% interval score', 'Expected interval score', 4)],
        sensitivity=[('PIP RMSE vs exact', 'PIP RMSE'), ('Mean excess marginal FKL vs exact', 'Excess marginal KL'),
                     ('Exact posterior mass in 95% latent interval', 'Coverage'),
                     ('Exact-posterior expected 95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_training_alpha_iterations_runtime.csv',
        paired=[('PIP RMSE vs exact', 'Δ PIP RMSE vs exact'), ('Mean excess marginal FKL vs exact', 'Δ Mean marginal KL vs exact'),
                ('Exact-posterior expected 95% interval score', 'Δ Exact-posterior expected 95% interval score')]),
    'riboflavin': dict(
        label='riboflavin', replicates='train--test splits', vi_k=False,
        cols=[('Predictive RMSE', 'RMSE', 4), ('MLPD', 'MLPD', 4), (K_COL, r'Mean $\widehat k$', 3),
              (ESS_COL, 'Mean ESS', 0), ('95% predictive coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('Predictive RMSE', 'Predictive RMSE'), ('MLPD', 'MLPD'),
                     ('95% predictive coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_training_alpha_iterations_runtime.csv',
        paired=[('Predictive RMSE', 'Δ RMSE'), ('MLPD', '−Δ MLPD'), ('95% interval score', 'Δ 95% interval score')]),
}


def _fmt(value, decimals):
    if not np.isfinite(value):
        return '--'
    return f'{value:.{decimals}f}' if decimals else f'{value:.0f}'


def _finite_mean(x):
    x = pd.to_numeric(pd.Series(x), errors='coerce').to_numpy(float)
    ok = np.isfinite(x)
    return (float(x[ok].mean()) if ok.any() else np.nan,
            float(x[ok].std(ddof=1)) if ok.sum() > 1 else np.nan, int(ok.sum()))


def _write_tabular(lines_body, header, ncols, path, note):
    lines = [r'\thesistablestyle',
             r'\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}ll' + 'r' * (ncols - 2) + '@{}}',
             r'\toprule', ' & '.join(header) + r'\\', r'\midrule'] + lines_body
    lines += [r'\bottomrule', r'\end{tabular*}']
    if note:
        lines += [r'\par\medskip', r'\begin{minipage}{\linewidth}\footnotesize', 'Note. ' + note, r'\end{minipage}']
    Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')


# ----------------------------------------------------------------------------- main table
def main_results_table(setting, run, out):
    spec = SETTINGS[setting]
    data = pd.read_csv(run / 'selected_configuration_metrics.csv')
    if ESS_COL not in data:
        raise KeyError(f'{ESS_COL} is missing from selected_configuration_metrics.csv')
    rows = []
    for stage in TABLE_STAGES:
        for method in TABLE_METHODS:
            g = data[(data.stage == stage) & (data.method == method)]
            if g.empty:
                raise ValueError(f'No selected results for {method} / {stage}')
            row = dict(stage=stage, method=method, replicates=g.seed.nunique())
            for col, _, _ in spec['cols']:
                source = g
                if col == K_COL and stage == 'Original VI' and spec['vi_k']:
                    source = data[(data.stage == 'Original PSIS') & (data.method == method)]
                mean, sd, n = _finite_mean(source[col])
                if stage == 'Original VI' and (col == ESS_COL or (col == K_COL and not spec['vi_k'])):
                    mean, sd, n = np.nan, np.nan, 0
                row[col], row[col + ' SD'], row[col + ' n_defined'] = mean, sd, n
            rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_main_results_with_ESS.csv', index=False)
    total = int(data.seed.nunique())
    body = []
    for i, stage in enumerate(TABLE_STAGES):
        block = table[table.stage == stage]
        for j, method in enumerate(block.method):
            cells = []
            for col, _, dp in spec['cols']:
                value = block.loc[block.method == method, col].iloc[0]
                n = block.loc[block.method == method, col + ' n_defined'].iloc[0]
                cell = _fmt(value, dp)
                if col == K_COL and np.isfinite(value) and n < total:
                    cell += f' ({int(n)})'
                cells.append(cell)
            first = (r'\multirow{5}{*}{' + STAGE_TEX[stage] + '}') if j == 0 else ''
            body.append(f'{first} & {METHOD_TEX[method]} & ' + ' & '.join(cells) + r'\\')
        if i < len(TABLE_STAGES) - 1:
            body.append(r'\midrule')
    header = ['Procedure', 'Method'] + [h for _, h, _ in spec['cols']]
    note = (('Original VI and Ordinary PSIS share the VI proposal and therefore the same $\\widehat k$. '
             if spec['vi_k'] else 'The $\\widehat k$ under Ordinary PSIS describes the Original VI proposal. ')
            + f'ESS is averaged over the {total} {spec["replicates"]}; Original VI has no importance weights. '
            + f'Parentheses give the number of defined $\\widehat k$ values when fewer than {total}.')
    _write_tabular(body, header, len(header), out / f'{setting}_main_results_with_ESS.tex', note)
    return table


# ----------------------------------------------------------------------------- selected training records
def selected_training_table(setting, run, out):
    spec = SETTINGS[setting]
    d = pd.read_csv(run / spec['selected_training'])
    iters = next(c for c in ('iterations', 'iters', 'training_iterations') if c in d)
    time_col = next(c for c in ('training_runtime_seconds', 'training_time_seconds', 'time (s)') if c in d)
    rows = []
    for method in TABLE_METHODS:
        g = d[d.method == method]
        if g.empty:
            continue
        if method == 'ELBO':
            alpha = np.nan
        elif 'annealed' in method:
            alpha = pd.to_numeric(g.final_alpha, errors='coerce').mean()
        else:
            alpha = pd.to_numeric(g.alpha_setting, errors='coerce').mean()
        rows.append(dict(method=method, n=len(g), mean_alpha=alpha,
                         mean_iterations=pd.to_numeric(g[iters]).mean(),
                         mean_selected_runtime_seconds=pd.to_numeric(g[time_col]).mean()))
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_selected_alpha_iterations_runtime.csv', index=False)
    return table


# ----------------------------------------------------------------------------- learning-rate table
def _main_allocation(df):
    for c in ['seed', 'learning_rate', 'S', 'N']:
        df[c] = pd.to_numeric(df[c], errors='raise')
    is_elbo = df['method'].eq('ELBO')
    return df.loc[(is_elbo & df['S'].eq(300) & df['N'].eq(1)) | (~is_elbo & df['S'].eq(1) & df['N'].eq(300))].copy()


def learning_rate_table(setting, run, out):
    df = _main_allocation(pd.read_csv(run / SETTINGS[setting]['evaluated']))
    value = 'evaluated_elbo_2000'
    df['_other'] = list(zip(df['alpha_setting'].astype(str), df['S'].astype(int), df['N'].astype(int)))
    if df.duplicated(['method', 'seed', 'learning_rate', '_other']).any():
        raise ValueError('Duplicate tuning configurations found.')
    for method, group in df.groupby('method'):
        grids = [frozenset(zip(g['seed'], g['_other'])) for _, g in group.groupby('learning_rate')]
        if any(grid != grids[0] for grid in grids[1:]):
            raise ValueError(f'{method}: unbalanced learning-rate grid')
    per_replicate = df.groupby(['method', 'seed', 'learning_rate'], as_index=False)[value].mean()
    summary = per_replicate.groupby(['method', 'learning_rate'], as_index=False)[value].mean()
    summary['best'] = summary[value].eq(summary.groupby('method')[value].transform('max'))
    summary.to_csv(out / f'{setting}_learning_rate_average_evaluated_ELBO.csv', index=False)
    best = summary[summary.best][['method', 'learning_rate', value]]
    best.to_csv(out / f'{setting}_best_learning_rate.csv', index=False)
    return best


# ----------------------------------------------------------------------------- configuration sensitivity
def configuration_sensitivity_table(setting, run, out):
    spec = SETTINGS[setting]
    d = pd.read_csv(run / spec['all_metrics'])
    d = d[d.stage.eq('Original VI')]
    if 'S' in d:
        d = _main_allocation(d)
    rows = []
    for method in TABLE_METHODS:
        g = d[d.method.eq(method)]
        row = dict(method=method)
        for col, name in spec['sensitivity']:
            x = g[['seed', col]].copy()
            x[col] = pd.to_numeric(x[col], errors='coerce')
            by_seed = x.groupby('seed')[col].agg(['mean', lambda v: v.std(ddof=1)])
            by_seed.columns = ['mean', 'sd']
            row[name] = by_seed['mean'].mean()
            row[name + ' (configuration SD)'] = by_seed['sd'].mean()
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_original_VI_configuration_sensitivity.csv', index=False)
    return table


# ----------------------------------------------------------------------------- figures
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
LABEL = {'Damped PSIS': 'Iterative Damped PSIS'}
PAIRED_COLS = ['oriented_mean_difference_left_is_better', 'oriented_95% interval lower', 'oriented_95% interval upper']


def _save(fig, out, name):
    for ext in ['png', 'pdf']:
        fig.savefig(out / f'{name}.{ext}', dpi=300, bbox_inches='tight', pad_inches=.12)
    plt.close(fig)


def _clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)
    ax.xaxis.label.set_size(12)
    ax.yaxis.label.set_size(12)


def paired_plot(setting, run, out, only=False):
    specs = SETTINGS[setting]['paired']
    df = pd.read_csv(run / 'paired_comparisons_vs_ELBO.csv')
    df.stage = df.stage.replace({'Iterative Damped PSIS': 'Damped PSIS'})
    stages = ['Damped PSIS'] if only else TABLE_STAGES
    if df.duplicated(['method', 'stage', 'metric']).any():
        raise ValueError('Duplicate paired rows.')
    if len(specs) == 5:
        fig = plt.figure(figsize=(16.2, 8.8)); grid = fig.add_gridspec(2, 6, hspace=.55, wspace=.55)
        axes = [fig.add_subplot(grid[0, :3]), fig.add_subplot(grid[0, 3:]), fig.add_subplot(grid[1, :2]),
                fig.add_subplot(grid[1, 2:4]), fig.add_subplot(grid[1, 4:])]
    else:
        fig, axes = plt.subplots(1, 3, figsize=(16.2, 5.6))
    for i, (ax, (metric, title)) in enumerate(zip(axes, specs)):
        bounds = []
        for stage in stages:
            j = TABLE_STAGES.index(stage)
            dd = df[(df.metric == metric) & (df.stage == stage)].set_index('method').loc[TABLE_METHODS[1:]]
            x, lo, hi = dd[PAIRED_COLS].apply(pd.to_numeric, errors='raise').to_numpy(float).T
            good = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
            y = np.arange(4) + (0 if only else [-.27, -.09, .09, .27][j])
            color = '#2ca02c' if only else COLORS[j]
            ax.hlines(y[good], lo[good], hi[good], color=color, lw=1.1)
            ax.plot(x[good], y[good], linestyle='none', marker='D' if only else ['o', 's', '^', 'D'][j],
                    color=color, ms=5.3, label=LABEL.get(stage, stage))
            ax.vlines(lo[good], y[good] - .035, y[good] + .035, color=color, lw=1)
            ax.vlines(hi[good], y[good] - .035, y[good] + .035, color=color, lw=1)
            bounds.extend(np.r_[lo[good], hi[good], x[good]])
        bound = max(np.max(np.abs(bounds)) * 1.12 if bounds else .001, 1e-8)
        ax.set_xlim(-bound, bound); ax.axvline(0, color='.35', ls='--', lw=1)
        ax.set_yticks(range(4), TABLE_METHODS[1:]); ax.set_ylim(3.55, -.55)
        ax.set_title(f'({chr(97 + i)}) {title}', loc='left', fontsize=10)
        ax.set_xlabel('Paired difference vs ELBO'); _clean_axes(ax)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5)); ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
        if i not in ([0, 2] if len(specs) == 5 else [0]):
            ax.tick_params(axis='y', labelleft=False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(stages), frameon=False, bbox_to_anchor=(.5, .025))
    fig.subplots_adjust(left=.11, right=.99, top=.88 if len(specs) == 5 else .80,
                        bottom=.16 if len(specs) == 5 else .20, wspace=.38)
    _save(fig, out, f'{setting}-paired' + ('-iterative-damped-only' if only else ''))


def _p1000_pips(run, gibbs_dir):
    selected = pd.read_csv(run / 'selected_configurations.csv'); rows = []
    for seed, g in selected.groupby('seed'):
        seed = int(seed); meta = pd.read_csv(gibbs_dir / f'runtime_and_metrics_gibbs_seed_{seed:02d}.csv').iloc[0]
        refs = []
        for chain in range(1, int(meta.n_chains) + 1):
            with np.load(gibbs_dir / f'gibbs_draws_seed_{seed:02d}_chain_{chain}.npz', allow_pickle=False) as z:
                refs.append(z['samples'].mean(0))
        ref = np.mean(refs, axis=0)
        for m in g.method:
            token = m.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()
            d = pd.read_csv(run / 'selected_pips' / f'seed_{seed:02d}_{token}.csv')
            for s in TABLE_STAGES:
                rows.append(pd.DataFrame({'seed': seed, 'method': m, 'stage': s, 'reference_pip': ref,
                                          'method_pip': d['pip_' + s.lower().replace(' ', '_')]}))
    return pd.concat(rows, ignore_index=True)


def pip_scatter(setting, run, out, gibbs_dir=None, varbvs=None):
    if setting == 'p1000':
        pips = _p1000_pips(run, gibbs_dir)
    else:
        pips = pd.read_csv(run / 'selected_pips_all_stages.csv')
    pips.stage = pips.stage.replace({'Iterative Damped PSIS': 'Damped PSIS'})
    if varbvs is not None:
        pips = pd.concat([pips, varbvs], ignore_index=True)
    ref = 'exact' if setting == 'bodyfat' else 'Gibbs'
    for stage in TABLE_STAGES:
        df = pips[pips.stage.eq(stage)]
        methods = TABLE_METHODS + (['varbvs'] if stage == 'Original VI' and 'varbvs' in set(df.method) else [])
        fig, axes = plt.subplots(2, 3, figsize=(15.5, 9.2))
        for ax, m in zip(axes.flat, methods):
            dd = df[df.method.eq(m)]
            ax.scatter(dd.reference_pip, dd.method_pip, s=14, alpha=.38, color=COLORS[0])
            ax.plot([0, 1], [0, 1], '--', lw=1, color=COLORS[0])
            ax.set(xlim=(-.03, 1.03), ylim=(-.03, 1.03), title=m,
                   xlabel=f'{"Exact" if ref == "exact" else ref} posterior inclusion probability',
                   ylabel='Estimated posterior inclusion probability')
            _clean_axes(ax)
        for ax in list(axes.flat)[len(methods):]:
            ax.axis('off')
        fig.tight_layout(rect=[0, .02, 1, .97])
        suffix = '' if stage == 'Original VI' else '-' + LABEL.get(stage, stage).lower().replace(' ', '-')
        _save(fig, out, setting + '-pips' + suffix)
    # Figure 4.2: annealed VIMCO-GM across the four stages
    sub = pips[pips.method.eq('VIMCO-GM annealed')]
    fig, axes = plt.subplots(2, 2, figsize=(12.8, 10.0))
    for ax, stage in zip(axes.flat, ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']):
        dd = sub[sub.stage.eq(stage)]
        ax.scatter(dd.reference_pip, dd.method_pip, s=14, alpha=.38, color=COLORS[0])
        ax.plot([0, 1], [0, 1], '--', lw=1, color=COLORS[0])
        ax.set(xlim=(-.03, 1.03), ylim=(-.03, 1.03), title=stage,
               xlabel=f'{"Exact" if ref == "exact" else ref} posterior inclusion probability',
               ylabel='Estimated posterior inclusion probability')
        _clean_axes(ax)
    fig.tight_layout(rect=[0, .02, 1, .97])
    _save(fig, out, setting + '-postprocessing-pips')


# ----------------------------------------------------------------------------- p = 30 extras
def _holm(pvalues):
    values = np.asarray(pvalues, dtype=float); result = np.full(len(values), np.nan)
    valid = np.where(np.isfinite(values))[0]; ranked = valid[np.argsort(values[valid])]; largest = 0.0
    for rank, idx in enumerate(ranked):
        largest = max(largest, min(1.0, float(values[idx]) * (len(ranked) - rank))); result[idx] = largest
    return result


def _repeated_measures_f(x):
    n, k = x.shape
    contrasts = x - x[:, :1]
    centered = contrasts - contrasts.mean(axis=1, keepdims=True)
    treatment = centered.mean(axis=0)
    ss_s = float(n * np.sum(treatment ** 2)); error = centered - treatment[None, :]
    ss_error = float(np.sum(error ** 2)); df_s, df_e = k - 1, (n - 1) * (k - 1)
    F_value = (ss_s / df_s) / (ss_error / df_e)
    cov = np.cov(centered, rowvar=False, ddof=1)
    eps = float(np.clip(np.trace(cov) ** 2 / (df_s * np.sum(cov * cov)), 1.0 / df_s, 1.0))
    return dict(datasets=n, F=float(F_value), df1_GG=eps * df_s, df2_GG=eps * df_e, GG_epsilon=eps,
                p_uncorrected=float(f_distribution.sf(F_value, df_s, df_e)),
                p_GG=float(f_distribution.sf(F_value, eps * df_s, eps * df_e)))


def _select_two_mcse(group):
    """Evaluated-ELBO rule of Section 4.2.1: among configurations within two combined MCSEs of
    the largest evaluated ELBO, take the one with the smallest MCSE."""
    e, m = 'evaluated_elbo_2000', 'evaluated_elbo_2000 MCSE'
    top = group.loc[group[e].idxmax()]
    tied = group[(top[e] - group[e]) <= 2 * np.sqrt(top[m] ** 2 + group[m] ** 2)]
    return tied.sort_values(m).iloc[0]


def allocation_f_tests(run, out):
    """Table B.1: for every dataset, method and allocation (S, N), the learning rate (and alpha for
    the fixed variants) is selected by the evaluated-ELBO / MCSE rule, exactly as in the main protocol.
    The evaluated ELBO of the selected fits is compared across the five allocations with a one-way
    repeated-measures ANOVA (Greenhouse-Geisser corrected, Holm across the four methods). The same
    test is reported for the PIP RMSE of the selected fits (Original VI)."""
    d = pd.read_csv(run / 'evaluated_configurations.csv')
    levels = [(1, 300), (3, 100), (6, 50), (10, 30), (30, 10)]
    d = d[d.method.isin(TABLE_METHODS[1:]) & d.S.isin([s for s, _ in levels])]
    selected = pd.DataFrame([_select_two_mcse(g) for _, g in d.groupby(['method', 'seed', 'S', 'N'])])
    metrics = pd.read_csv(run / 'all_configuration_metrics_full_grid.csv')
    metrics = metrics[metrics.stage.eq('Original VI')].drop_duplicates('config_id')
    selected = selected.merge(metrics[['config_id', 'PIP RMSE vs Gibbs']], on='config_id', how='left')
    selected.to_csv(out / 'p30_allocation_selected_fits.csv', index=False)
    tables = {}
    for value, stem in [('evaluated_elbo_2000', 'evaluated_ELBO'), ('PIP RMSE vs Gibbs', 'PIP_RMSE')]:
        means, tests = [], []
        for method in TABLE_METHODS[1:]:
            wide = selected[selected.method.eq(method)].pivot(index='seed', columns='S', values=value)
            wide = wide[[s for s, _ in levels]]
            tests.append(dict(method=method, **_repeated_measures_f(wide.to_numpy(float))))
            means.append(dict(method=method, **{f'S={s}, N={n}': wide[s].mean() for s, n in levels}))
        tests = pd.DataFrame(tests); tests['p_GG_Holm'] = _holm(tests.p_GG)
        table = pd.DataFrame(means).merge(tests[['method', 'F', 'df1_GG', 'df2_GG', 'p_GG_Holm']], on='method')
        table.to_csv(out / f'p30_allocation_selected_{stem}_and_F_tests.csv', index=False)
        tables[stem] = table
    return tables


def varbvs_summary(path, out):
    """Table 4.3 varbvs row: PIP RMSE and excess marginal forward KL against the
    varbvs-matched Gibbs reference, averaged over the 50 datasets."""
    d = pd.read_csv(path)
    eps = 1e-12
    z = d.gibbs_pip.clip(eps, 1 - eps); q = d.varbvs_pip.clip(eps, 1 - eps)
    d['kl'] = z * np.log(z / q) + (1 - z) * np.log((1 - z) / (1 - q))
    per = d.groupby('replicate').agg(rmse=('squared_pip_error', lambda v: np.sqrt(v.mean())), fkl=('kl', 'mean'))
    per.to_csv(out / 'p30_varbvs_vs_matched_gibbs_by_replicate.csv')
    result = pd.DataFrame([dict(method='varbvs', **{'PIP RMSE': per.rmse.mean(), 'Delta FKL': per.fkl.mean()})])
    result.to_csv(out / 'p30_varbvs_row.csv', index=False)
    scatter = pd.DataFrame({'seed': d.replicate, 'method': 'varbvs', 'stage': 'Original VI',
                            'reference_pip': d.gibbs_pip, 'method_pip': d.varbvs_pip})
    return result, scatter


# ----------------------------------------------------------------------------- driver
def make_thesis_outputs(setting, results_dir, gibbs_dir=None, varbvs_csv=None):
    matplotlib.rcParams['text.usetex'] = False
    run = Path(results_dir).resolve()
    out = run / 'THESIS_TABLES_AND_FIGURES'; out.mkdir(exist_ok=True)
    produced = {}
    produced['main'] = main_results_table(setting, run, out)
    produced['training'] = selected_training_table(setting, run, out)
    produced['learning_rate'] = learning_rate_table(setting, run, out)
    produced['sensitivity'] = configuration_sensitivity_table(setting, run, out)
    varbvs_scatter = None
    if setting == 'p30':
        produced['allocation'] = allocation_f_tests(run, out)
        if varbvs_csv is not None and Path(varbvs_csv).is_file():
            produced['varbvs'], varbvs_scatter = varbvs_summary(varbvs_csv, out)
        else:
            warnings.warn('varbvs comparison CSV not found; varbvs row and panel skipped.')
    paired_plot(setting, run, out)
    paired_plot(setting, run, out, only=True)
    if setting in ('p30', 'p1000', 'bodyfat'):
        pip_scatter(setting, run, out, gibbs_dir=gibbs_dir)
    (out / 'README.txt').write_text(__doc__)
    print('Thesis tables and figures saved in', out)
    return produced


In [ ]:
THESIS_OUTPUTS = make_thesis_outputs('p1000', RESULTS_DIR, gibbs_dir=ROOT / 'output_gibbs_tau2_1')